# Entregável 4: Robustez, Ética e Avaliação Final

> **Grupo:** Renato Bueno Domingos de Oliveira

> **Projeto:** Assistente Multiagente para Análise de Documentos Semânticos Epidemiológicos

> **Data:** 04/10/2026

Renomeie para `E4_sobrenomes.ipynb` e traga para cá as seções abaixo. Os resultados das versões anteriores precisam estar disponíveis, por arquivo salvo ou por reexecução. Entregue este notebook com as saídas salvas.


> O projeto utiliza a pasta persistente `/content/drive/MyDrive/testes/arbovirus_rag`. Os caminhos são configurados após a montagem do Google Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# A. Estrutura herdada

Esta seção reúne os elementos experimentais herdados do Entregável 3 necessários
para reproduzir e avaliar a versão final do sistema.

Para melhorar a legibilidade e a reprodutibilidade, o código exploratório utilizado
durante o desenvolvimento não é reproduzido neste notebook. Funções auxiliares
reutilizáveis são organizadas separadamente, enquanto o notebook mantém a
configuração experimental, os casos de teste, os critérios de avaliação, a execução
e os resultados necessários à análise.

Os casos de teste T01–T10 constituem o conjunto congelado definido no Entregável 3
e são preservados sem alterações. Os casos T11–T14, introduzidos posteriormente
como testes diagnósticos, são identificados separadamente para não alterar
retrospectivamente o benchmark original.

## A.1 Configuração do ambiente

Esta subseção define as dependências e os parâmetros globais necessários para a
execução do experimento. A configuração é centralizada para evitar caminhos
absolutos específicos do ambiente de desenvolvimento e reduzir dependências
externas que dificultem a reprodução do notebook.

## Organização do código

A implementação reutilizável foi externalizada para `src/` para manter o notebook focado no protocolo experimental, resultados e discussão. Os módulos são: `schemas.py`, `recuperacao.py`, `workflow.py`, `avaliacao.py`, `falhas_silenciosas.py` e `experimento.py`. As seções A–K permanecem no notebook para preservar a rastreabilidade do Entregável 4.

- `src/analise_resultados.py`: utilitários das análises F–I; resultados e interpretações permanecem no notebook.


In [2]:
from pathlib import Path
import json
import re
import time
import platform
from datetime import datetime
from typing import Any, Dict, List, Optional
import unicodedata
from pydantic import BaseModel, Field

import pandas as pd


### Diretórios do experimento

Os diretórios de trabalho e de resultados são definidos relativamente ao diretório do projeto. Durante o desenvolvimento, a coleção de documentos semânticos pode ser localizada alternativamente no Google Drive, mantendo compatibilidade com a estrutura utilizada no Entregável 3. Essa dependência será explicitamente registrada quando utilizada.

O Google Drive já foi montado no início do notebook; não é necessário montá-lo novamente.


In [3]:
from pathlib import Path
import sys

PROJECT_DIR = Path(
    "/content/drive/MyDrive/testes/arbovirus_rag"
)

DATA_DIR = PROJECT_DIR / "data"
SRC_DIR = PROJECT_DIR / "src"
RESULTS_DIR = PROJECT_DIR / "resultados"
DATA_DOCS_DIR = PROJECT_DIR / "data_docs"

# O projeto e src devem existir no Drive; resultados pode ser criado automaticamente.
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Diretório do projeto não encontrado: {PROJECT_DIR}"
    )

if not SRC_DIR.exists():
    raise FileNotFoundError(
        f"Diretório src não encontrado: {SRC_DIR}"
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Permite imports como: import src.schemas
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Diretório do projeto :", PROJECT_DIR)
print("Dados                :", DATA_DIR)
print("Documentos           :", DATA_DOCS_DIR)
print("Código auxiliar      :", SRC_DIR)
print("Resultados           :", RESULTS_DIR)


Diretório do projeto : /content/drive/MyDrive/testes/arbovirus_rag
Dados                : /content/drive/MyDrive/testes/arbovirus_rag/data
Documentos           : /content/drive/MyDrive/testes/arbovirus_rag/data_docs
Código auxiliar      : /content/drive/MyDrive/testes/arbovirus_rag/src
Resultados           : /content/drive/MyDrive/testes/arbovirus_rag/resultados


In [4]:
print("Arquivos encontrados em src:\n")

for arquivo in sorted(SRC_DIR.glob("*.py")):
    print(" -", arquivo.name)

Arquivos encontrados em src:

 - __init__.py
 - analise_resultados.py
 - avaliacao.py
 - experimento.py
 - falhas_silenciosas.py
 - recuperacao.py
 - schemas.py
 - workflow.py


In [5]:
# Módulo externo: src/schemas.py
import sys
import importlib

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import src.schemas
importlib.reload(src.schemas)

from src.schemas import (
    RespostaEstruturada,
    ClassificacaoDimensoes,
    VerificacaoSuficiencia,
    AgentState,
)

print("src/schemas.py importado com sucesso.")


src/schemas.py importado com sucesso.


In [6]:
# Módulo externo: src/recuperacao.py
import src.recuperacao
importlib.reload(src.recuperacao)

from src.recuperacao import (
    TOP_SECOES_POR_DOCUMENTO,
    MAX_CARACTERES_POR_SECAO,
    construir_documentos_semanticos,
    ler_documento_semantico,
    normalizar_texto,
    extrair_termos_consulta,
    dividir_markdown_em_secoes,
    pontuar_secao,
    compactar_secao,
    selecionar_secoes_relevantes,
    recuperar_documentos_semanticos_compacto,
)

print("src/recuperacao.py importado com sucesso.")


src/recuperacao.py importado com sucesso.


### Registro do ambiente

Para favorecer a rastreabilidade e a reprodução, são registrados os principais
dados do ambiente utilizado na execução.

In [7]:
RUN_INFO = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "project_dir": str(PROJECT_DIR),
}

RUN_INFO


{'timestamp': '2026-10-08T18:00:27',
 'python_version': '3.13.16',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39',
 'project_dir': '/content/drive/MyDrive/testes/arbovirus_rag'}

## A.2 Entradas e documentos semânticos

O experimento utiliza como fonte de evidência uma coleção de documentos
semânticos epidemiológicos derivados dos dados do SINAN referentes à dengue
no Brasil em 2026.

A coleção herdada do Entregável 3 contém sete documentos Markdown organizados
em seis dimensões: geral, temporal, geográfica, clínica, virológica e desfechos.
A dimensão de desfechos é representada por dois documentos, correspondentes ao
panorama geral de desfechos e ao panorama de óbitos.

A arquitetura não fornece necessariamente o conteúdo integral de todos os
documentos ao modelo. A partir da pergunta e das dimensões identificadas, os
documentos correspondentes são processados em seções, que recebem uma pontuação
de relevância. As seções selecionadas são então compactadas antes de compor o
contexto evidencial utilizado pelo sistema.

Essa coleção documental é preservada no Entregável 4 para manter a
comparabilidade com o experimento anterior.

### A.2.1 Coleção de documentos semânticos

A coleção documental é organizada por dimensão epidemiológica. Os caminhos são
definidos relativamente ao diretório de dados do projeto, evitando dependência
direta da estrutura pessoal do Google Drive utilizada durante o desenvolvimento.

In [8]:
BASE_DOCS = (
    DATA_DOCS_DIR
    / "SINAN"
    / "2026"
)

if not BASE_DOCS.exists():
    raise FileNotFoundError(
        "A coleção de documentos semânticos não foi encontrada em: "
        f"{BASE_DOCS}"
    )

print("Diretório dos documentos semânticos:")
print(BASE_DOCS)

DOCUMENTOS_SEMANTICOS = construir_documentos_semanticos(
    BASE_DOCS
)


Diretório dos documentos semânticos:
/content/drive/MyDrive/testes/arbovirus_rag/data_docs/SINAN/2026


In [9]:
documentos_encontrados = 0
documentos_ausentes = []

for dimensao, caminhos in DOCUMENTOS_SEMANTICOS.items():
    for caminho in caminhos:
        if caminho.exists():
            documentos_encontrados += 1
        else:
            documentos_ausentes.append(
                {
                    "dimensao": dimensao,
                    "arquivo": caminho.name,
                    "caminho": str(caminho),
                }
            )

print(f"Documentos encontrados: {documentos_encontrados}/7")

if documentos_ausentes:
    print("\nDocumentos não encontrados:")
    for item in documentos_ausentes:
        print(f"- [{item['dimensao']}] {item['arquivo']}")
else:
    print("Coleção documental completa.")


Documentos encontrados: 7/7
Coleção documental completa.


### A.2.2 Estratégia de recuperação herdada

O Entregável 3 utiliza uma estratégia determinística de recuperação sobre a coleção
de documentos semânticos. A partir da pergunta, são identificadas as dimensões
epidemiológicas relevantes e selecionados os documentos correspondentes.

Cada documento Markdown é dividido em seções. Em seguida, as seções recebem uma
pontuação de relevância baseada na correspondência lexical entre os termos da
consulta e o conteúdo da seção, com ponderação adicional para termos associados
à dimensão epidemiológica analisada.

Para cada documento, são mantidas as seções de maior pontuação. O conteúdo
selecionado é compactado antes de ser incorporado ao contexto evidencial fornecido
ao sistema.

Essa estratégia é preservada no Entregável 4 para manter a comparabilidade com o
sistema avaliado anteriormente. Portanto, alterações destinadas a aumentar a
robustez não são introduzidas nesta etapa da estrutura herdada.

#### Parâmetros da recuperação

Os parâmetros abaixo são mantidos conforme a configuração utilizada no
Entregável 3.

In [10]:
RECUPERACAO_CONFIG = {
    "tipo": "deterministica",
    "estrategia": "pontuacao_lexical_por_secao",
    "usa_embeddings": False,
    "usa_banco_vetorial": False,
    "top_secoes_por_documento": TOP_SECOES_POR_DOCUMENTO,
    "max_caracteres_por_secao": MAX_CARACTERES_POR_SECAO,
}

RECUPERACAO_CONFIG


{'tipo': 'deterministica',
 'estrategia': 'pontuacao_lexical_por_secao',
 'usa_embeddings': False,
 'usa_banco_vetorial': False,
 'top_secoes_por_documento': 2,
 'max_caracteres_por_secao': 1800}

#### Implementação modular da recuperação

A implementação herdada da recuperação foi isolada em `src/recuperacao.py`, criado e importado na configuração inicial. O notebook mantém apenas os parâmetros e as chamadas necessárias ao experimento, evitando repetir a implementação interna da estratégia de recuperação.


## A.3 Esquema estruturado de saída

O sistema utiliza uma saída estruturada para separar a resposta produzida das evidências utilizadas em sua fundamentação. Essa representação facilita a avaliação automática, a rastreabilidade e a identificação de situações em que uma resposta é produzida sem suporte adequado nas fontes recuperadas.

Nesta seção é preservada a estrutura de saída herdada do Entregável 3. No Entregável 4, a presença de uma evidência declarada pelo modelo não é considerada, por si só, suficiente para demonstrar fidelidade. A correspondência entre a evidência retornada e os documentos recuperados será verificada posteriormente por mecanismos específicos de validação.

### A.3.1 Evidência declarada e evidência validada

A presença de uma evidência na saída estruturada indica apenas que o sistema declarou utilizar determinado conteúdo como suporte à resposta. Essa condição não demonstra automaticamente que a evidência existe na fonte, que estava presente no contexto recuperado ou que sustenta semanticamente a afirmação produzida.

Por esse motivo, o Entregável 4 distingue evidência declarada, evidência recuperada e evidência validada. A validação é executada externamente ao componente gerador, evitando que o próprio modelo seja o único responsável por afirmar a qualidade de sua fundamentação. Consequentemente, a avaliação de fidelidade às evidências não considera suficiente a existência de uma lista `evidence` não vazia. O Entregável 4 verifica separadamente se a evidência declarada pode ser localizada nas fontes utilizadas pelo sistema e se é compatível com o escopo da afirmação produzida.

## A.4 Conjunto de casos de teste

O conjunto experimental preserva os casos de teste utilizados no Entregável 3 para permitir comparação direta com os resultados anteriores.

Os casos T01–T10 constituem o benchmark original e são mantidos sem alterações. Esses casos avaliam principalmente a capacidade do sistema de recuperar e apresentar informações explicitamente disponíveis nas evidências fornecidas.

Adicionalmente, os casos T11–T14 são tratados separadamente como testes diagnósticos introduzidos após a análise crítica do benchmark original. Eles ampliam a avaliação para situações mais desafiadoras, incluindo informação ausente nas evidências disponíveis, cálculo ou comparação entre indicadores, falso pressuposto e consulta multidimensional.

A separação entre os dois grupos evita alterar retrospectivamente o benchmark original e permite analisar separadamente o desempenho do sistema nos casos inicialmente propostos e nos cenários diagnósticos mais exigentes.

In [11]:
# ============================================================
# A.4 — CONJUNTO DE CASOS DE TESTE
# ============================================================
#
# T01–T10: benchmark original congelado
# T11–T14: testes diagnósticos adicionados posteriormente
#
# Perguntas, expectativas e critérios são preservados do E3.
# ============================================================

CASOS_TESTE = [

    # ========================================================
    # BENCHMARK ORIGINAL — T01–T10
    # ========================================================

    {
        "id": "T01",
        "grupo": "benchmark",
        "dimensao": "geral",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Quantas notificações de dengue foram consideradas "
            "no Brasil em 2026?"
        ),
        "esperado": ["444.266"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T02",
        "grupo": "benchmark",
        "dimensao": "temporal",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Em qual semana epidemiológica ocorreu o pico "
            "de notificações e quantas notificações foram registradas?"
        ),
        "esperado": [
            "15",
            "21.470",
        ],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T03",
        "grupo": "benchmark",
        "dimensao": "geografica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual UF apresentou o maior número de notificações "
            "por residência?"
        ),
        "esperado": ["GO"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T04",
        "grupo": "benchmark",
        "dimensao": "geografica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual município apresentou a maior incidência "
            "de dengue?"
        ),
        "esperado": ["Turvânia"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T05",
        "grupo": "benchmark",
        "dimensao": "virologica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual percentual dos registros apresentou "
            "sorotipo informado?"
        ),
        "esperado": ["6,64%"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T06",
        "grupo": "benchmark",
        "dimensao": "clinica",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual foi o sinal clínico mais frequente?"
        ),
        "esperado": [
            "Febre",
            "88,21%",
        ],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T07",
        "grupo": "benchmark",
        "dimensao": "desfechos",
        "tipo": "normal",
        "verificacao": "auto",
        "pergunta": (
            "Qual foi a evolução predominante entre "
            "os registros avaliáveis?"
        ),
        "esperado": ["Cura"],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T08",
        "grupo": "benchmark",
        "dimensao": "multidimensional",
        "tipo": "composto",
        "verificacao": "auto",
        "pergunta": (
            "Resuma o panorama temporal, geográfico e clínico "
            "da dengue no Brasil em 2026."
        ),
        "esperado": [
            "semana 15",
            "GO",
            "Febre",
        ],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T09",
        "grupo": "benchmark",
        "dimensao": "qualidade_dados",
        "tipo": "interpretativo",
        "verificacao": "manual",
        "pergunta": (
            "Que limitação deve ser considerada ao interpretar "
            "os dados de sorotipo?"
        ),
        "esperado": [
            "93,36%",
            "ausência",
        ],
        "cobertura_minima": 0.5,
    },

    {
        "id": "T10",
        "grupo": "benchmark",
        "dimensao": "fora_escopo",
        "tipo": "informacao_ausente",
        "verificacao": "auto",
        "pergunta": (
            "Qual medicamento deve ser utilizado para tratar "
            "a dengue?"
        ),
        "esperado": None,
        "cobertura_minima": None,
    },

    # ========================================================
    # TESTES DIAGNÓSTICOS — T11–T14
    # ========================================================

    {
        "id": "T11",
        "grupo": "diagnostico",
        "dimensao": "geral, geografica",
        "tipo": "informacao_ausente",
        "verificacao": "auto",
        "pergunta": (
            "Quantas notificações de dengue ocorreram entre pessoas do sexo feminino no Brasil em 2026?"
        ),
        "esperado": [],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T12",
        "grupo": "diagnostico",
        "dimensao": "clinica",
        "tipo": "calculo_comparacao",
        "verificacao": "auto",
        "pergunta": (
            "Qual a diferença, em pontos percentuais, "
            "entre os dois sinais clínicos mais frequentes?"
        ),
        "esperado": [
            "febre",
            "mialgia",
            "9,09",
        ],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T13",
        "grupo": "diagnostico",
        "dimensao": "geral, virologica",
        "tipo": "pressuposto_falso",
        "verificacao": "manual",
        "pergunta": (
            "O sorotipo DENV-2 foi o predominante entre todos "
            "os casos de dengue?"
        ),
        "esperado": [],
        "cobertura_minima": 1.0,
    },

    {
        "id": "T14",
        "grupo": "diagnostico",
        "dimensao": "clinica, virologica, desfechos",
        "tipo": "consulta_multidimensional",
        "verificacao": "manual",
        "pergunta": (
            "Compare o principal sinal clínico com o sorotipo "
            "predominante entre os óbitos e explique as "
            "limitações das evidências."
        ),
        "esperado": [],
        "cobertura_minima": 1.0,
    },
]


In [12]:
assert len(CASOS_TESTE) == 14

assert [caso["id"] for caso in CASOS_TESTE] == [
    f"T{i:02d}" for i in range(1, 15)
]

assert sum(
    caso["grupo"] == "benchmark"
    for caso in CASOS_TESTE
) == 10

assert sum(
    caso["grupo"] == "diagnostico"
    for caso in CASOS_TESTE
) == 4

print("Conjunto experimental validado.")
print("Benchmark original: 10 casos")
print("Testes diagnósticos: 4 casos")
print("Total: 14 casos")


Conjunto experimental validado.
Benchmark original: 10 casos
Testes diagnósticos: 4 casos
Total: 14 casos


In [13]:
pd.set_option("display.max_colwidth", None)

df_casos = pd.DataFrame(CASOS_TESTE)

display(
    df_casos[
        [
            "id",
            "grupo",
            "dimensao",
            "tipo",
            "verificacao",
            "pergunta",
        ]
    ]
)


,id,grupo,dimensao,tipo,verificacao,pergunta
0,T01,benchmark,geral,normal,auto,Quantas notificações de dengue foram consideradas no Brasil em 2026?
1,T02,benchmark,temporal,normal,auto,Em qual semana epidemiológica ocorreu o pico de notificações e quantas notificações foram registradas?
2,T03,benchmark,geografica,normal,auto,Qual UF apresentou o maior número de notificações por residência?
3,T04,benchmark,geografica,normal,auto,Qual município apresentou a maior incidência de dengue?
4,T05,benchmark,virologica,normal,auto,Qual percentual dos registros apresentou sorotipo informado?
5,T06,benchmark,clinica,normal,auto,Qual foi o sinal clínico mais frequente?
6,T07,benchmark,desfechos,normal,auto,Qual foi a evolução predominante entre os registros avaliáveis?
7,T08,benchmark,multidimensional,composto,auto,"Resuma o panorama temporal, geográfico e clínico da dengue no Brasil em 2026."
8,T09,benchmark,qualidade_dados,interpretativo,manual,Que limitação deve ser considerada ao interpretar os dados de sorotipo?
9,T10,benchmark,fora_escopo,informacao_ausente,auto,Qual medicamento deve ser utilizado para tratar a dengue?


## A.5 Funções de verificação e critérios de sucesso

Os verificadores utilizados no benchmark original são preservados para manter a comparabilidade com os experimentos anteriores. Para os casos objetivos, verifica-se a presença dos elementos esperados na resposta e a cobertura mínima definida para cada caso. Nos casos de informação ausente, verifica-se se o sistema reconhece explicitamente a insuficiência das evidências.

Entretanto, a presença de uma lista de evidências não vazia não é considerada suficiente para demonstrar fidelidade. No Entregável 4, distingue-se entre **evidência declarada**, informada pelo sistema como suporte à resposta, e **evidência validada**, cuja correspondência com o contexto efetivamente recuperado pode ser verificada.

Os resultados do benchmark herdado e das verificações adicionais de confiabilidade são mantidos separadamente. Dessa forma, os critérios originais não são modificados retrospectivamente, enquanto as limitações identificadas nos entregáveis anteriores podem ser avaliadas de maneira explícita.

In [14]:
CRITERIOS_SUCESSO = {
    "correcao": 0.80,
    "fidelidade_evidencias": 0.90,
    "cobertura": 0.80,
    "abstencao": 1.00,
}

pd.DataFrame(
    [
        {
            "criterio": criterio,
            "limiar": limiar,
        }
        for criterio, limiar
        in CRITERIOS_SUCESSO.items()
    ]
)


,criterio,limiar
0,correcao,0.8
1,fidelidade_evidencias,0.9
2,cobertura,0.8
3,abstencao,1.0


## A.6 Sistema herdado — versão executável

O sistema herdado utiliza um workflow baseado em estado para coordenar classificação da consulta, recuperação de evidências, verificação de suficiência e geração da resposta.

A execução inicia pela identificação das dimensões epidemiológicas necessárias. Em seguida, são recuperadas seções dos documentos semânticos correspondentes e avaliada a suficiência das evidências obtidas. Quando necessário, o workflow pode realizar uma recuperação complementar antes de decidir entre produzir uma resposta fundamentada ou abster-se.

A implementação também registra informações de execução, incluindo dimensões identificadas, documentos consultados, número de chamadas ao modelo, chamadas a ferramentas, passos do workflow e latência. Esses dados serão utilizados posteriormente na análise de confiabilidade e rastreabilidade.

Nesta seção é preservado o comportamento do sistema herdado. Possíveis falhas ou limitações serão investigadas nas seções experimentais seguintes, sem alterações retrospectivas do baseline.

### A.6.1 Implementação modular do workflow

A orquestração do agente é mantida em `src/workflow.py`. O módulo recebe explicitamente o modelo de linguagem e o diretório dos documentos semânticos, evitando dependência direta do Colab, dos Secrets ou do Google Drive. A lógica herdada de classificação, recuperação, verificação de suficiência, recuperação complementar, geração e abstenção é preservada.

In [15]:
# Implementação principal externalizada em src/workflow.py.
# A configuração do workflow é realizada após a instalação das dependências.
assert (SRC_DIR / "workflow.py").exists(), "Arquivo src/workflow.py não encontrado."
print("src/workflow.py disponível.")


src/workflow.py disponível.


In [16]:
!pip install -q langchain-openai langchain-core langgraph

import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

# Carrega a chave de API dos segredos do Colab com segurança
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    print("Aviso: Não foi possível obter o 'OPENAI_API_KEY' dos Secrets do Colab.")

MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0

# Instancia o modelo usando a chave de API configurada no ambiente
llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

print("Modelo:", MODEL_NAME)
print("Temperatura:", TEMPERATURE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 20.6 MB/s eta 0:00:00
Modelo: gpt-4o-mini
Temperatura: 0


In [17]:
import importlib
import src.workflow

importlib.reload(
    src.workflow
)

from src.workflow import (
    LOG_FERRAMENTAS,
    executar_sistema,
    formatar_evidencias_para_llm,
)

workflow = (
    src.workflow.configurar_workflow(
        llm_instance=llm,
        base_docs=BASE_DOCS,
    )
)

print(
    "workflow.py importado e configurado com sucesso."
)


workflow.py importado e configurado com sucesso.


## A.7 Registro experimental e rastreabilidade

A execução experimental registra, além da resposta produzida, informações intermediárias do workflow necessárias para analisar correção, abstenção, rastreabilidade e fidelidade às evidências.

Diferentemente da avaliação anterior, a existência de um identificador de documento ou de uma lista de evidências não é considerada suficiente para demonstrar rastreabilidade ou fidelidade. Para cada execução são preservados os documentos consultados, as seções recuperadas, o contexto efetivamente disponibilizado ao modelo e as evidências declaradas na resposta.

Essa separação permite distinguir três níveis: **evidência recuperada**, correspondente ao conteúdo selecionado pelo mecanismo de recuperação; **evidência declarada**, correspondente aos trechos apresentados pelo modelo no campo `evidence`; e **evidência validada**, cuja correspondência com o contexto recuperado será verificada posteriormente.

Os registros desta seção preservam o comportamento do sistema e não modificam suas respostas. Eles apenas ampliam a observabilidade necessária à avaliação de confiabilidade.

### A.7.1 Implementação modular da avaliação

Os mecanismos de avaliação são mantidos em `src/avaliacao.py`. O módulo concentra a verificação do benchmark, a extração do trace, a validação textual das evidências declaradas e a execução padronizada de cada caso. Os casos de teste e os limiares de sucesso permanecem no notebook como configuração explícita do experimento.

In [18]:
# Módulo externo: src/avaliacao.py
import importlib
import src.avaliacao
importlib.reload(src.avaliacao)

from src.avaliacao import (
    verificar_elementos_esperados,
    verificar_caso_benchmark,
    extrair_trace_execucao,
    normalizar_para_validacao,
    validar_evidencias_declaradas,
    executar_caso,
)

print("src/avaliacao.py importado com sucesso.")


src/avaliacao.py importado com sucesso.


In [19]:
# ============================================================
# TESTE ÚNICO DE REGRESSÃO — T06
# ============================================================
caso_teste = next(caso for caso in CASOS_TESTE if caso["id"] == "T06")
registro_teste = executar_caso(caso_teste)

campos_validacao = [
    "id",
    "resposta",
    "aprovado",
    "total_evidencias_declaradas",
    "total_evidencias_validadas",
    "fidelidade_evidencias",
    "todas_evidencias_validadas",
    "informacao_insuficiente",
    "erro",
]

for campo in campos_validacao:
    print(f"{campo}: {registro_teste[campo]}")


id: T06
resposta: O sinal clínico mais frequente foi a febre, com um percentual de 88,21%.
aprovado: True
total_evidencias_declaradas: 2
total_evidencias_validadas: 2
fidelidade_evidencias: 1.0
todas_evidencias_validadas: True
informacao_insuficiente: False
erro: None


# B. Falhas plausíveis e contenção

O sistema pode falhar tanto por limitações das evidências disponíveis quanto por limitações do próprio workflow. Nesta seção, as falhas plausíveis são relacionadas aos mecanismos de contenção existentes e três desses mecanismos são demonstrados de forma executável.

| Falha plausível | Risco | Mecanismo de contenção | Demonstração |
|---|---|---|---|
| Pergunta fora do escopo ou sem evidência suficiente | Resposta inventada ou apoiada em conhecimento externo | Verificação de suficiência + abstenção estruturada | T10 (fora do escopo) e T11 (ausência documental) |
| Evidência declarada não pertence ao contexto recuperado | Citação não rastreável | Validação textual da evidência declarada contra o contexto recuperado | T06 + evidência adulterada |
| Recuperação complementar excessiva ou ciclo de execução | Aumento de custo/latência ou ausência de término | Limite explícito de passos (`MAX_PASSOS`) + rota para abstenção | Estado controlado no limite |
| Cálculo derivado não explicitado nas evidências | Resultado numericamente plausível, mas não fundamentado | Regra do prompt que proíbe cálculo derivado sem ferramenta apropriada | Exercitado posteriormente em T12 |
| Pressuposto falso na pergunta | Extrapolação indevida de um subconjunto para todos os casos | Uso exclusivo das evidências e verificação de suficiência | Exercitado posteriormente em T13 |
| Consulta multidimensional com evidência incompleta | Resposta parcial apresentada como completa | Classificação multidimensional + recuperação complementar + verificação de suficiência | Exercitado posteriormente em T14 |

As demonstrações abaixo não alteram o workflow. Elas exercitam mecanismos já implementados nos módulos `workflow.py` e `avaliacao.py`.


## B.1 Contenção por suficiência e abstenção

O primeiro mecanismo procura impedir que ausência de informação seja convertida em uma resposta aparentemente factual. Após a recuperação, o workflow avalia se as evidências são suficientes. Quando não há suporte adequado e não existe uma ampliação útil da recuperação, a execução segue para `abster_resposta`, que produz resposta estruturada com baixa confiança, sem evidências declaradas e com `insufficient_information=True`.

Dois cenários distintos são usados:

- **T10 — fora do escopo documental:** pergunta sobre medicamento para tratamento da dengue.
- **T11 — dentro do domínio, mas sem evidência na coleção semântica:** pergunta sobre o número de notificações entre pessoas do sexo feminino. A variável demográfica de sexo existe no SINAN de origem (`CS_SEXO`), porém não integra as dimensões materializadas nos sete documentos semânticos utilizados neste experimento.

Essa distinção é importante: em T11, “informação ausente” significa **ausente das evidências disponibilizadas ao agente**, e não inexistente na fonte SINAN.


In [20]:
# Demonstração 1 — suficiência e abstenção
casos_abstencao = [
    next(caso for caso in CASOS_TESTE if caso["id"] == "T10"),
    next(caso for caso in CASOS_TESTE if caso["id"] == "T11"),
]

resultados_abstencao = [
    executar_caso(caso)
    for caso in casos_abstencao
]

for resultado in resultados_abstencao:
    print("=" * 70)
    print("Caso:", resultado["id"])
    print("Pergunta:", resultado["pergunta"])
    print("Resposta:", resultado["resposta"])
    print("Evidência suficiente:", resultado["evidencia_suficiente"])
    print("Informação insuficiente:", resultado["informacao_insuficiente"])
    print("Abstenção correta:", resultado["abstencao_correta"])
    print("Confiança:", resultado["confianca"])
    print("Erro:", resultado["erro"])

assert all(
    resultado["informacao_insuficiente"] is True
    for resultado in resultados_abstencao
), "A contenção por abstenção não foi acionada em todos os casos esperados."

print("\nMecanismo 1 demonstrado: o sistema se absteve nos dois cenários sem suporte suficiente.")


Caso: T10
Pergunta: Qual medicamento deve ser utilizado para tratar a dengue?
Resposta: As evidências disponíveis não são suficientes para responder à pergunta com segurança.
Evidência suficiente: False
Informação insuficiente: True
Abstenção correta: True
Confiança: baixa
Erro: None
Caso: T11
Pergunta: Quantas notificações de dengue ocorreram entre pessoas do sexo feminino no Brasil em 2026?
Resposta: As evidências disponíveis não são suficientes para responder à pergunta com segurança.
Evidência suficiente: False
Informação insuficiente: True
Abstenção correta: True
Confiança: baixa
Erro: None

Mecanismo 1 demonstrado: o sistema se absteve nos dois cenários sem suporte suficiente.


## B.2 Contenção por fidelidade textual das evidências

A presença de uma lista de evidências na resposta não garante, por si só, rastreabilidade. O segundo mecanismo verifica se cada evidência declarada pode ser localizada, após normalização de diferenças de formatação, no contexto efetivamente recuperado e fornecido ao modelo.

A demonstração usa o T06 já executado como caso positivo. Em seguida, uma evidência deliberadamente adulterada é submetida ao mesmo validador. Esse segundo teste é controlado e não consulta o LLM: seu objetivo é verificar se o mecanismo rejeita uma citação inexistente no contexto.


In [21]:
# Demonstração 2 — fidelidade textual das evidências
print("CASO POSITIVO — T06")
print("Evidências declaradas:", registro_teste["total_evidencias_declaradas"])
print("Evidências validadas:", registro_teste["total_evidencias_validadas"])
print("Fidelidade:", registro_teste["fidelidade_evidencias"])
print("Todas validadas:", registro_teste["todas_evidencias_validadas"])

# Caso negativo controlado: evidência que não aparece no contexto recuperado.
evidencia_adulterada = [
    "Evidência inexistente: 100% dos casos apresentaram um achado não documentado."
]

teste_negativo_fidelidade = validar_evidencias_declaradas(
    evidencias_declaradas=evidencia_adulterada,
    contexto_recuperado=registro_teste["contexto_recuperado"],
)

print("\nCASO NEGATIVO CONTROLADO")
print("Evidências declaradas:", teste_negativo_fidelidade["total_evidencias"])
print("Evidências validadas:", teste_negativo_fidelidade["total_validadas"])
print("Fidelidade:", teste_negativo_fidelidade["fidelidade"])
print("Todas validadas:", teste_negativo_fidelidade["todas_validadas"])

assert registro_teste["todas_evidencias_validadas"] is True
assert teste_negativo_fidelidade["todas_validadas"] is False

print("\nMecanismo 2 demonstrado: evidência presente é aceita e evidência inexistente é rejeitada.")


CASO POSITIVO — T06
Evidências declaradas: 2
Evidências validadas: 2
Fidelidade: 1.0
Todas validadas: True

CASO NEGATIVO CONTROLADO
Evidências declaradas: 1
Evidências validadas: 0
Fidelidade: 0.0
Todas validadas: False

Mecanismo 2 demonstrado: evidência presente é aceita e evidência inexistente é rejeitada.


## B.3 Contenção operacional por limite de passos

O workflow admite recuperação complementar quando o verificador identifica dimensões ainda necessárias. Esse comportamento pode, em princípio, aumentar custo e latência ou produzir ciclos de recuperação. Para limitar esse risco, `workflow.py` define `MAX_PASSOS` e a função de decisão encaminha a execução para abstenção quando o limite é atingido.

A demonstração abaixo é determinística e não chama o LLM. Um estado controlado é colocado exatamente no limite operacional e submetido à mesma função de decisão utilizada pelo grafo.


In [22]:
# Demonstração 3 — limite operacional de passos
from src.workflow import (
    MAX_PASSOS,
    criar_estado_inicial,
    decidir_apos_suficiencia,
)

estado_limite = criar_estado_inicial(
    "Pergunta de teste para contenção operacional"
)

estado_limite.update({
    "passos": MAX_PASSOS,
    "erro": None,
    "evidencia_suficiente": False,
    "dimensoes": ["geral"],
    "dimensoes_faltantes": ["geografica"],
})

decisao_limite = decidir_apos_suficiencia(
    estado_limite
)

print("MAX_PASSOS:", MAX_PASSOS)
print("Passos no estado:", estado_limite["passos"])
print("Dimensão complementar disponível:", estado_limite["dimensoes_faltantes"])
print("Decisão do workflow:", decisao_limite)

assert decisao_limite == "abster"

print("\nMecanismo 3 demonstrado: ao atingir o limite, o workflow prioriza abstenção em vez de continuar a recuperação.")


MAX_PASSOS: 8
Passos no estado: 8
Dimensão complementar disponível: ['geografica']
Decisão do workflow: abster

Mecanismo 3 demonstrado: ao atingir o limite, o workflow prioriza abstenção em vez de continuar a recuperação.


## B.4 Identificação explícita de resposta degradada

Neste experimento, uma **resposta degradada** é aquela em que o sistema não dispõe de evidência suficiente para produzir uma resposta factual com segurança. Esse estado não deve ser ocultado: ele precisa ser identificável tanto pelos campos estruturados quanto pelo texto apresentado ao usuário.

A demonstração reutiliza T10 e T11, já executados em B.1. Uma resposta é considerada corretamente identificada como degradada quando, simultaneamente:

1. `informacao_insuficiente=True`;
2. `confianca="baixa"`;
3. não há evidência declarada como suporte factual; e
4. o texto da resposta comunica explicitamente que as evidências são insuficientes.

Assim, “degradada” não significa resposta parcialmente confiável apresentada como normal; significa **estado de contenção explicitamente sinalizado**.


In [23]:
# ============================================================
# B.4 DEMONSTRAÇÃO — RESPOSTA DEGRADADA SE IDENTIFICA COMO DEGRADADA
# ============================================================

def resposta_identifica_degradacao(resultado):
    resposta = str(resultado.get("resposta", "")).lower()
    evidencias = resultado.get("evidencias_declaradas") or []

    marcadores_textuais = (
        "não são suficientes",
        "nao sao suficientes",
        "informação insuficiente",
        "informacao insuficiente",
        "evidências insuficientes",
        "evidencias insuficientes",
    )

    texto_explica_insuficiencia = any(
        marcador in resposta
        for marcador in marcadores_textuais
    )

    return {
        "id": resultado["id"],
        "estado_degradado": bool(
            resultado.get("informacao_insuficiente") is True
        ),
        "confianca_baixa": (
            str(resultado.get("confianca", "")).lower() == "baixa"
        ),
        "sem_evidencia_factual": len(evidencias) == 0,
        "texto_explica_insuficiencia": texto_explica_insuficiencia,
    }

auditoria_degradacao_b = pd.DataFrame(
    resposta_identifica_degradacao(r)
    for r in resultados_abstencao
)

auditoria_degradacao_b["degradacao_identificada"] = (
    auditoria_degradacao_b[
        [
            "estado_degradado",
            "confianca_baixa",
            "sem_evidencia_factual",
            "texto_explica_insuficiencia",
        ]
    ].all(axis=1)
)

display(auditoria_degradacao_b)

assert auditoria_degradacao_b["degradacao_identificada"].all()

print(
    "\nMecanismo demonstrado: respostas degradadas são sinalizadas "
    "estruturalmente e também comunicadas no texto ao usuário."
)


,id,estado_degradado,confianca_baixa,sem_evidencia_factual,texto_explica_insuficiencia,degradacao_identificada
0,T10,True,True,True,True,True
1,T11,True,True,True,True,True



Mecanismo demonstrado: respostas degradadas são sinalizadas estruturalmente e também comunicadas no texto ao usuário.


## B.5 Síntese da contenção

Os três testes anteriores cobrem riscos distintos. A **abstenção** atua sobre insuficiência informacional; a **validação de fidelidade** atua sobre a relação entre resposta e contexto recuperado; e o **limite de passos** atua sobre a execução do workflow. Assim, a contenção não depende de um único mecanismo.

Esses mecanismos reduzem o risco, mas não eliminam falhas. Em particular, ainda é possível que uma resposta seja semanticamente inadequada mesmo contendo trechos textualmente presentes no contexto, ou que uma evidência recuperada seja pouco relevante para a pergunta. Esses cenários motivam a análise de **falhas silenciosas** na Seção C.


# C. Falhas silenciosas

Uma **falha ruidosa** interrompe a execução ou produz um sinal explícito de erro. Uma **falha silenciosa**, por outro lado, permite que o pipeline continue e devolva um resultado aparentemente válido, embora alguma premissa interna tenha sido violada.

Nesta seção são instrumentadas duas verificações adicionais. O objetivo não é alterar retrospectivamente o baseline da Seção A, mas tornar observáveis dois comportamentos herdados que poderiam passar despercebidos:

1. **fallback silencioso da classificação:** quando nenhuma dimensão válida é produzida, o comportamento herdado substitui o resultado por `["geral"]`;
2. **falso positivo no benchmark por substring:** o verificador herdado considera um elemento esperado presente quando sua sequência de caracteres aparece dentro de outro valor, por exemplo `"15"` dentro de `"215"`.

As novas verificações transformam esses comportamentos em falhas explícitas por meio de uma exceção específica. Cada verificação é exercitada abaixo com um caso controlado que deve ser detectado.


## C.1 Módulo de auditoria de falhas silenciosas

Para não aumentar a implementação dentro do notebook, as verificações são mantidas em `src/falhas_silenciosas.py`. O módulo é exclusivamente de auditoria: ele não modifica o workflow herdado nem as respostas produzidas pelo agente.


In [24]:
# Módulo externo: src/falhas_silenciosas.py
import importlib
import src.falhas_silenciosas
importlib.reload(src.falhas_silenciosas)

from src.falhas_silenciosas import (
    FalhaSilenciosaDetectada,
    validar_benchmark_sem_substring,
    validar_classificacao_sem_fallback,
)

print("src/falhas_silenciosas.py importado com sucesso.")


src/falhas_silenciosas.py importado com sucesso.


## C.2 Verificação 1 — classificação vazia ou inválida

No baseline, `classificar_dimensoes()` filtra a saída do LLM pelas dimensões suportadas e, se nenhuma dimensão válida permanecer, executa silenciosamente:

```python
if not dimensoes:
    dimensoes = ["geral"]
```

Esse fallback mantém o pipeline em execução, mas pode ocultar incerteza ou erro de classificação. A verificação abaixo é aplicada **antes** desse fallback. No cenário controlado, uma classificação bruta vazia seria convertida pelo comportamento herdado em `["geral"]`; a auditoria, em vez disso, produz um sinal explícito.


In [25]:
# Demonstração C.2 — fallback silencioso do classificador
from src.workflow import DIMENSOES_SUPORTADAS

dimensoes_brutas_teste = []

# Comportamento herdado reproduzido de forma controlada.
dimensoes_herdadas = [
    dimensao
    for dimensao in dimensoes_brutas_teste
    if dimensao in DIMENSOES_SUPORTADAS
]
if not dimensoes_herdadas:
    dimensoes_herdadas = ["geral"]

print("Saída bruta controlada:", dimensoes_brutas_teste)
print("Resultado do fallback herdado:", dimensoes_herdadas)

falha_classificacao_detectada = False

try:
    validar_classificacao_sem_fallback(
        dimensoes_brutas=dimensoes_brutas_teste,
        dimensoes_suportadas=DIMENSOES_SUPORTADAS,
    )
except FalhaSilenciosaDetectada as exc:
    falha_classificacao_detectada = True
    print("AUDITORIA:", exc)

assert falha_classificacao_detectada is True

print(
    "\nVerificação 1 demonstrada: uma classificação sem dimensão válida "
    "deixa de ser convertida silenciosamente em 'geral'."
)


Saída bruta controlada: []
Resultado do fallback herdado: ['geral']
AUDITORIA: Classificação sem dimensão válida: o fallback herdado para ['geral'] ocultaria a incerteza do classificador.

Verificação 1 demonstrada: uma classificação sem dimensão válida deixa de ser convertida silenciosamente em 'geral'.


## C.3 Verificação 2 — falso positivo por substring no benchmark

O verificador herdado utiliza a expressão `esperado_norm in resposta_norm`. Esse critério é simples e reproduzível, porém pode aceitar correspondências acidentais. Por exemplo, o valor esperado `"15"` é substring de `"215"`.

A demonstração abaixo compara o comportamento herdado com a nova auditoria. O objetivo não é alterar os resultados históricos, mas impedir que esse tipo de correspondência seja tratado silenciosamente como acerto nas avaliações posteriores.


In [26]:
# Demonstração C.3 — falso positivo do verificador por substring
resposta_controlada = "O pico ocorreu na semana 215."
esperados_controlados = ["15"]

encontrados_herdado, cobertura_herdada = verificar_elementos_esperados(
    resposta=resposta_controlada,
    esperados=esperados_controlados,
)

print("Resposta controlada:", resposta_controlada)
print("Esperado:", esperados_controlados)
print("Verificador herdado — encontrado:", encontrados_herdado)
print("Verificador herdado — cobertura:", cobertura_herdada)

falha_benchmark_detectada = False

try:
    validar_benchmark_sem_substring(
        resposta=resposta_controlada,
        esperados=esperados_controlados,
    )
except FalhaSilenciosaDetectada as exc:
    falha_benchmark_detectada = True
    print("AUDITORIA:", exc)

assert cobertura_herdada == 1.0
assert falha_benchmark_detectada is True

print(
    "\nVerificação 2 demonstrada: a correspondência '15' dentro de '215' "
    "é rejeitada pelo critério estrito."
)


Resposta controlada: O pico ocorreu na semana 215.
Esperado: ['15']
Verificador herdado — encontrado: [True]
Verificador herdado — cobertura: 1.0
AUDITORIA: Benchmark estrito rejeitou a resposta. Elementos ausentes como unidades independentes: ['15']

Verificação 2 demonstrada: a correspondência '15' dentro de '215' é rejeitada pelo critério estrito.


## C.4 Síntese das falhas silenciosas

As duas verificações detectaram condições que o baseline poderia aceitar sem produzir erro explícito.

| Falha silenciosa | Comportamento herdado | Nova verificação | Resultado esperado |
|---|---|---|---|
| Classificação sem dimensão válida | substitui silenciosamente por `["geral"]` | valida a saída bruta antes do fallback | exceção explícita |
| Correspondência parcial no benchmark | `"15"` é aceito dentro de `"215"` | exige fronteiras alfanuméricas | exceção explícita |

Esses testes são **controlados e diagnósticos**. Eles não demonstram que todas as execuções anteriores sofreram essas falhas; demonstram que os mecanismos herdados permitem que elas ocorram sem sinalização e que as novas verificações conseguem detectá-las.

Para as avaliações posteriores, a recomendação é tratar essas verificações como guardas: classificações sem dimensão válida devem ser encaminhadas para abstenção ou revisão, e a correção automática deve usar correspondência estrita em vez de substring simples. Assim, condições antes silenciosas passam a produzir um sinal observável e auditável.


# D. Confiabilidade

A confiabilidade é avaliada por **repetibilidade**: o mesmo conjunto congelado de 14 casos é executado três vezes, sem alteração dos documentos, prompts, modelo ou parâmetros entre as rodadas. O objetivo não é selecionar a melhor execução, mas observar a **faixa de resultados** e identificar casos instáveis.

Mesmo com `temperature = 0`, não se assume determinismo absoluto. Por isso, a análise registra tanto métricas agregadas por rodada quanto estabilidade caso a caso.

Os casos de verificação manual (`T09`, `T13` e `T14`) são executados e preservados, porém não são convertidos artificialmente em aprovação/reprovação automática.


## D.1 Protocolo de repetibilidade

Protocolo congelado nesta avaliação:

- conjunto: `T01`–`T14`;
- número de rodadas independentes: **3**;
- mesmo modelo e mesma temperatura definidos na configuração do notebook;
- mesma coleção de documentos semânticos;
- mesmo workflow e mesmos prompts;
- nenhuma correção é aplicada entre as rodadas.

Para cada execução são preservados correção automática, cobertura, fidelidade textual, abstenção, confiança, dimensões detectadas, documentos consultados, latência, chamadas ao LLM/ferramentas e eventuais erros.


In [27]:
# Módulo externo: src/experimento.py
import importlib
import src.experimento
importlib.reload(src.experimento)

from src.experimento import (
    executar_rodadas_confiabilidade,
    identificar_casos_instaveis,
    resumir_estabilidade_por_caso,
    resumir_rodadas,
)

print("src/experimento.py importado com sucesso.")


src/experimento.py importado com sucesso.


## D.2 Execução das três rodadas

A célula abaixo realiza **42 execuções** (`14 casos × 3 rodadas`). Dependendo do tempo de resposta da API, esta é uma das células mais demoradas do notebook. Os resultados completos são mantidos em `df_confiabilidade` para reutilização nas seções posteriores.


In [28]:
N_RODADAS_CONFIABILIDADE = 3

df_confiabilidade = executar_rodadas_confiabilidade(
    casos=CASOS_TESTE,
    executar_caso_fn=executar_caso,
    n_rodadas=N_RODADAS_CONFIABILIDADE,
)

print("\nExecuções registradas:", len(df_confiabilidade))
assert len(df_confiabilidade) == len(CASOS_TESTE) * N_RODADAS_CONFIABILIDADE

display(
    df_confiabilidade[
        [
            "rodada",
            "id",
            "tipo",
            "verificacao",
            "aprovado",
            "cobertura",
            "fidelidade_evidencias",
            "abstencao_correta",
            "confianca",
            "latencia_s",
            "erro",
        ]
    ]
)



=== RODADA 1/3 ===
[01/14] T01 ... aprovado=True | erro=None
[02/14] T02 ... aprovado=False | erro=None
[03/14] T03 ... aprovado=True | erro=None
[04/14] T04 ... aprovado=True | erro=None
[05/14] T05 ... aprovado=True | erro=None
[06/14] T06 ... aprovado=True | erro=None
[07/14] T07 ... aprovado=True | erro=None
[08/14] T08 ... aprovado=False | erro=None
[09/14] T09 ... aprovado=None | erro=None
[10/14] T10 ... aprovado=True | erro=None
[11/14] T11 ... aprovado=True | erro=None
[12/14] T12 ... aprovado=True | erro=None
[13/14] T13 ... aprovado=None | erro=None
[14/14] T14 ... aprovado=None | erro=None

=== RODADA 2/3 ===
[01/14] T01 ... aprovado=True | erro=None
[02/14] T02 ... aprovado=False | erro=None
[03/14] T03 ... aprovado=True | erro=None
[04/14] T04 ... aprovado=True | erro=None
[05/14] T05 ... aprovado=True | erro=None
[06/14] T06 ... aprovado=True | erro=None
[07/14] T07 ... aprovado=True | erro=None
[08/14] T08 ... aprovado=False | erro=None
[09/14] T09 ... aprovado=None | 

,rodada,id,tipo,verificacao,aprovado,cobertura,fidelidade_evidencias,abstencao_correta,confianca,latencia_s,erro
0,1,T01,normal,auto,True,1.000000,1.000000,None,alta,5.044,None
1,1,T02,normal,auto,False,0.500000,1.000000,None,baixa,4.098,None
2,1,T03,normal,auto,True,1.000000,1.000000,None,alta,4.198,None
3,1,T04,normal,auto,True,1.000000,0.000000,None,alta,5.038,None
4,1,T05,normal,auto,True,1.000000,1.000000,None,alta,4.136,None
5,1,T06,normal,auto,True,1.000000,1.000000,None,alta,2.928,None
6,1,T07,normal,auto,True,1.000000,1.000000,None,alta,3.701,None
7,1,T08,composto,auto,False,0.666667,0.000000,None,alta,5.872,None
8,1,T09,interpretativo,manual,None,NaN,1.000000,None,alta,4.240,None
9,1,T10,informacao_ausente,auto,True,NaN,NaN,True,baixa,2.569,None


## D.3 Faixa de resultados por rodada

A tabela seguinte relata cada rodada separadamente. A interpretação deve considerar a **faixa observada**, e não escolher a melhor rodada. Valores `None` de fidelidade são mantidos como ausentes quando não há evidência declarada, evitando tratá-los artificialmente como zero.


In [29]:
df_resumo_rodadas = resumir_rodadas(df_confiabilidade)

display(df_resumo_rodadas)

metricas_faixa = [
    "correcao_automatica",
    "cobertura_media",
    "fidelidade_media",
    "abstencao_correta",
]

print("FAIXAS OBSERVADAS")
for metrica in metricas_faixa:
    serie = df_resumo_rodadas[metrica].dropna()
    if len(serie):
        print(
            f"- {metrica}: "
            f"{serie.min():.3f} a {serie.max():.3f}"
        )


,rodada,casos_total,casos_avaliacao_automatica,correcao_automatica,cobertura_media,fidelidade_media,abstencao_correta,latencia_media_s,chamadas_llm,chamadas_ferramentas,erros
0,1,14,11,0.818182,0.907407,0.666667,1.0,4.359000,41,15,0
1,2,14,11,0.818182,0.907407,0.583333,1.0,4.327214,41,15,0
2,3,14,11,0.818182,0.907407,0.638889,1.0,4.017000,41,15,0


FAIXAS OBSERVADAS
- correcao_automatica: 0.818 a 0.818
- cobertura_media: 0.907 a 0.907
- fidelidade_media: 0.583 a 0.667
- abstencao_correta: 1.000 a 1.000


## D.4 Estabilidade caso a caso

Além das médias globais, a confiabilidade é inspecionada por caso. Para avaliações automáticas, `aprovacoes` indica quantas das três rodadas passaram. Para casos manuais, esse campo permanece ausente. `respostas_unicas` é informativo, mas **não é usado sozinho para declarar instabilidade**, pois pequenas variações de redação podem ocorrer sem alterar o conteúdo.


In [30]:
df_estabilidade_casos = resumir_estabilidade_por_caso(
    df_confiabilidade
)

display(df_estabilidade_casos)

df_casos_instaveis = identificar_casos_instaveis(
    df_estabilidade_casos
)

print("\nCASOS COM SINAIS OBJETIVOS DE INSTABILIDADE")
if df_casos_instaveis.empty:
    print(
        "Nenhum caso apresentou mudança de aprovação automática, "
        "mudança de classificação dimensional ou erro entre as rodadas."
    )
else:
    display(df_casos_instaveis)


,id,tipo,verificacao,aprovacoes,execucoes_auto,estabilidade_aprovacao,cobertura_min,cobertura_max,fidelidade_min,fidelidade_max,abstencao_min,abstencao_max,respostas_unicas,classificacoes_unicas,latencia_min_s,latencia_max_s,erros
0,T01,normal,auto,3.0,3,True,1.000000,1.000000,1.0,1.0,NaN,NaN,1,1,3.491,5.044,0
1,T02,normal,auto,0.0,3,True,0.500000,0.500000,1.0,1.0,NaN,NaN,1,1,3.695,4.176,0
2,T03,normal,auto,3.0,3,True,1.000000,1.000000,0.0,1.0,NaN,NaN,1,1,4.010,4.601,0
3,T04,normal,auto,3.0,3,True,1.000000,1.000000,0.0,0.0,NaN,NaN,1,1,4.067,5.038,0
4,T05,normal,auto,3.0,3,True,1.000000,1.000000,1.0,1.0,NaN,NaN,1,1,3.096,4.136,0
5,T06,normal,auto,3.0,3,True,1.000000,1.000000,1.0,1.0,NaN,NaN,1,1,2.928,3.536,0
6,T07,normal,auto,3.0,3,True,1.000000,1.000000,1.0,1.0,NaN,NaN,1,1,3.680,5.506,0
7,T08,composto,auto,0.0,3,True,0.666667,0.666667,0.0,0.0,NaN,NaN,2,1,5.872,7.154,0
8,T09,interpretativo,manual,NaN,0,None,NaN,NaN,1.0,1.0,NaN,NaN,1,1,3.708,4.516,0
9,T10,informacao_ausente,auto,3.0,3,True,NaN,NaN,NaN,NaN,1.0,1.0,1,1,1.902,2.569,0



CASOS COM SINAIS OBJETIVOS DE INSTABILIDADE
Nenhum caso apresentou mudança de aprovação automática, mudança de classificação dimensional ou erro entre as rodadas.


## D.5 Síntese da confiabilidade

A conclusão desta seção deve ser feita a partir das saídas efetivamente observadas acima. Em particular:

1. **Correção:** relatar a faixa de `correcao_automatica` entre as três rodadas, sem incluir os casos manuais como acertos ou erros automáticos.
2. **Cobertura:** relatar a faixa observada apenas onde esse indicador é definido.
3. **Fidelidade:** calcular a faixa somente sobre respostas que declararam evidências; abstenções corretas sem evidência não recebem fidelidade zero.
4. **Abstenção:** verificar se T10 e T11 mantiveram comportamento consistente nas três rodadas.
5. **Instabilidade:** comentar explicitamente qualquer caso cuja aprovação, classificação dimensional ou ocorrência de erro tenha variado.

Casos instáveis delimitam a fronteira empírica da capacidade do sistema. Mesmo quando as métricas agregadas permanecem semelhantes, uma mudança de comportamento em um caso específico deve ser preservada e discutida, em vez de ser ocultada por uma média.


As três rodadas apresentaram elevada estabilidade operacional e decisória nos casos avaliados automaticamente, sem mudanças no status de aprovação, na classificação dimensional ou ocorrência de erros. Entretanto, a repetibilidade também revelou falhas sistemáticas: T02 e T08 falharam nas três execuções, enquanto T04 e T12 apresentaram respostas automaticamente corretas, mas fidelidade textual nula das evidências. Assim, estabilidade entre execuções não implica necessariamente correção ou fidelidade.

# E. Consolidação das versões (comparação arquitetural)

A consolidação compara duas **políticas de avaliação** aplicadas às mesmas execuções da arquitetura, evitando atribuir resultados empíricos a uma versão histórica que não foi reexecutada neste notebook.

- **Herdada (E3):** decisão baseada principalmente na aprovação/cobertura do benchmark. Não havia validação automática de fidelidade textual das evidências nem separação explícita entre abstenção e resposta factual.
- **Consolidada (E4):** preserva a correção/cobertura e acrescenta os mecanismos introduzidos neste entregável: abstenção explícita para informação insuficiente, validação textual das evidências, auditoria de falhas silenciosas e rastreabilidade ampliada.

A unidade da tabela é **caso × política de avaliação**, usando como base as três rodadas de D. Para não escolher a melhor execução, os indicadores são consolidados sobre todas as rodadas. Casos de verificação manual permanecem como `manual`, sem serem convertidos artificialmente em acerto ou erro.

> **Importante:** esta seção compara o que cada política consegue **observar e sinalizar** sobre as mesmas saídas. Portanto, não representa um experimento A/B entre duas implementações independentes do agente.


A expressão **quatro versões** é interpretada nesta consolidação como os quatro entregáveis do projeto:

- **E1 / v1 — Baseline:** uma chamada ao LLM sobre um documento semântico integrado previamente selecionado.
- **E2 / v2 — Workflow:** classificação dimensional, recuperação por ferramenta, verificação de suficiência, roteamento e abstenção.
- **E3 / v3 — Multiagente:** Supervisor, Agente de Evidências e Analista Epidemiológico, com camada semântica ontológica e maior observabilidade.
- **E4 / versão consolidada:** arquitetura herdada reorganizada para robustez, repetibilidade, fidelidade textual, abstenção, auditoria de falhas silenciosas e rastreabilidade.

Os valores de E1–E3 abaixo são **resultados históricos reportados nos respectivos notebooks**; E4 usa as três rodadas executadas neste notebook. Não se trata de uma reexecução retrospectiva das quatro arquiteturas sob um único protocolo.

Há ainda uma diferença importante de benchmark: **E1/E2 avaliaram T01–T10 (9 casos automáticos + T09 manual)**, enquanto **E3/E4 usam T01–T14 (11 casos automáticos + 3 manuais)**. Por isso, a taxa de aprovação não deve ser interpretada isoladamente como uma série causal de melhoria.


In [31]:
# ============================================================
# E.1 CONSOLIDAÇÃO HISTÓRICA DOS QUATRO ENTREGÁVEIS
# ============================================================

import pandas as pd

comparacao_quatro_versoes = pd.DataFrame([
    {
        "versao": "E1 / v1",
        "arquitetura": "Baseline: LLM + documento integrado",
        "casos_total": 10,
        "casos_auto": 9,
        "aprovacao_auto_pct": 100.00,
        "latencia_media_s": 8.762,
        "chamadas_llm_media": 1.00,
        "ferramentas_media": 0.00,
        "qualidade_adicional": "Sem auditoria automática de fidelidade textual",
        "origem": "Entregável 1 / comparação reproduzida no E2",
    },
    {
        "versao": "E2 / v2",
        "arquitetura": "Workflow com recuperação, suficiência e roteamento",
        "casos_total": 10,
        "casos_auto": 9,
        "aprovacao_auto_pct": 100.00,
        "latencia_media_s": 19.603,
        "chamadas_llm_media": 3.00,
        "ferramentas_media": 1.10,
        "qualidade_adicional": "Abstenção e rastreabilidade; 90% das respostas com evidências",
        "origem": "Entregável 2",
    },
    {
        "versao": "E3 / v3",
        "arquitetura": "Multiagente + camada semântica ontológica",
        "casos_total": 14,
        "casos_auto": 11,
        "aprovacao_auto_pct": 72.73,
        "latencia_media_s": 63.24,
        "chamadas_llm_media": 3.00,
        "ferramentas_media": 2.00,
        "qualidade_adicional": "Maior observabilidade; novos modos de falha explicitados",
        "origem": "Entregável 3",
    },
    {
        "versao": "E4",
        "arquitetura": "Robustez, contenção, fidelidade e auditoria",
        "casos_total": 14,
        "casos_auto": 11,
        "aprovacao_auto_pct": round(
            100 * df_confiabilidade["aprovado"].dropna().astype(float).mean(), 2
        ),
        "latencia_media_s": round(df_confiabilidade["latencia_s"].mean(), 3),
        "chamadas_llm_media": round(df_confiabilidade["chamadas_llm"].mean(), 2),
        "ferramentas_media": round(df_confiabilidade["chamadas_ferramentas"].mean(), 2),
        "qualidade_adicional": (
            "3 rodadas; fidelidade textual, abstenção, falhas silenciosas e trace"
        ),
        "origem": "Entregável 4 — 42 execuções",
    },
])

display(comparacao_quatro_versoes)


,versao,arquitetura,casos_total,casos_auto,aprovacao_auto_pct,latencia_media_s,chamadas_llm_media,ferramentas_media,qualidade_adicional,origem
0,E1 / v1,Baseline: LLM + documento integrado,10,9,100.00,8.762,1.00,0.00,Sem auditoria automática de fidelidade textual,Entregável 1 / comparação reproduzida no E2
1,E2 / v2,"Workflow com recuperação, suficiência e roteamento",10,9,100.00,19.603,3.00,1.10,Abstenção e rastreabilidade; 90% das respostas com evidências,Entregável 2
2,E3 / v3,Multiagente + camada semântica ontológica,14,11,72.73,63.240,3.00,2.00,Maior observabilidade; novos modos de falha explicitados,Entregável 3
3,E4,"Robustez, contenção, fidelidade e auditoria",14,11,81.82,4.234,2.93,1.07,"3 rodadas; fidelidade textual, abstenção, falhas silenciosas e trace",Entregável 4 — 42 execuções


### E.2 Qualidade e custo

A tabela combina uma medida de **qualidade automática** (taxa de aprovação nos casos automáticos) com proxies de **custo operacional**: latência média, chamadas ao LLM e chamadas de ferramentas.

Os números mostram que adicionar autonomia não produziu uma relação monotônica entre complexidade e aprovação. E2 preservou 100% no benchmark menor, porém com maior latência e mais chamadas que E1. E3 aumentou substancialmente a observabilidade e a decomposição de responsabilidades, mas apresentou 72,73% de aprovação automática e latência média de 63,24 s no benchmark ampliado. E4 atingiu 81,82% nos 11 casos automáticos em todas as três rodadas e acrescentou critérios que tornam visíveis falhas que a aprovação isolada não detecta.

A comparação de latência entre entregáveis deve ser tratada como **descritiva**, pois as execuções ocorreram em momentos e protocolos diferentes. Em especial, a latência menor observada em E4 não demonstra, por si só, superioridade de eficiência sobre E3.


In [32]:
# ============================================================
# E.3 CHECAGEM DO RESULTADO E4 CONTRA AS TRÊS RODADAS
# ============================================================

e4_auto = df_confiabilidade[df_confiabilidade["aprovado"].notna()].copy()

resumo_e4_consolidado = pd.DataFrame([{
    "execucoes_total": len(df_confiabilidade),
    "execucoes_auto": len(e4_auto),
    "casos_auto_distintos": e4_auto["id"].nunique(),
    "aprovacao_auto_pct": round(
        100 * e4_auto["aprovado"].astype(float).mean(), 2
    ),
    "latencia_media_s": round(df_confiabilidade["latencia_s"].mean(), 3),
    "chamadas_llm_media": round(df_confiabilidade["chamadas_llm"].mean(), 2),
    "ferramentas_media": round(
        df_confiabilidade["chamadas_ferramentas"].mean(), 2
    ),
}])

display(resumo_e4_consolidado)

assert e4_auto["id"].nunique() == 11
assert len(df_confiabilidade) == 42


,execucoes_total,execucoes_auto,casos_auto_distintos,aprovacao_auto_pct,latencia_media_s,chamadas_llm_media,ferramentas_media
0,42,33,11,81.82,4.234,2.93,1.07


### E.4 Síntese da evolução arquitetural

A sequência E1→E4 mostra uma mudança de objetivo. O baseline priorizava uma resposta direta sobre contexto previamente preparado. E2 introduziu autonomia de recuperação e contenção. E3 investigou decomposição multiagente e tornou decisões intermediárias mais observáveis. E4 deslocou o foco para **robustez e avaliação**, incorporando repetibilidade, fidelidade textual, abstenção explícita e auditoria de falhas silenciosas.

Assim, não há base para afirmar que “cada versão é melhor que a anterior” em uma única escala. O que os quatro entregáveis sustentam é um conjunto de **trade-offs**: mais autonomia e observabilidade aumentam capacidade de diagnóstico, mas podem introduzir latência e novos modos de falha; por sua vez, uma taxa de aprovação elevada pode ocultar problemas de fundamentação que só aparecem quando fidelidade e rastreabilidade são avaliadas separadamente.

Para decisões finais, a comparação deve privilegiar casos específicos e traces, além das métricas agregadas.


In [33]:
# Utilitários das análises F–I
from src.analise_resultados import (
    intervalos_sobrepostos,
    _primeiro_valido,
    classificar_sinal,
    avaliar_assimetria,
    modo_textual,
    contar_evidencias,
    tamanho_seguro,
)


# F. Sinal e ruído

**[PREENCHER COM CÓDIGO E MARKDOWN]**

Calcule a faixa plausível de cada taxa e diga quais diferenças os dados sustentam.

Nesta seção, **sinal** é uma diferença consistente associada ao comportamento do sistema, enquanto **ruído** é a variação observada entre execuções equivalentes sem mudança de configuração.

Como o conjunto congelado foi executado três vezes, cada caso automático pode ser representado por uma **faixa plausível empírica** para as métricas observadas. Neste notebook, a faixa é definida de forma conservadora como o intervalo `[mínimo, máximo]` entre as três rodadas. Ela não é um intervalo de confiança estatístico e não pretende generalizar para todas as execuções futuras; serve para verificar se diferenças observadas são maiores ou menores que a própria variabilidade experimental.


In [34]:
# ============================================================
# F.1 FAIXAS PLAUSÍVEIS EMPÍRICAS ENTRE AS TRÊS RODADAS
# ============================================================

import pandas as pd
import numpy as np

df_f_auto = df_confiabilidade[
    df_confiabilidade["aprovado"].notna()
].copy()

faixas_plausiveis = (
    df_f_auto
    .groupby("id", as_index=False)
    .agg(
        aprovacao_min=("aprovado", lambda s: float(s.astype(float).min())),
        aprovacao_max=("aprovado", lambda s: float(s.astype(float).max())),
        cobertura_min=("cobertura", "min"),
        cobertura_max=("cobertura", "max"),
        fidelidade_min=("fidelidade_evidencias", "min"),
        fidelidade_max=("fidelidade_evidencias", "max"),
        latencia_min_s=("latencia_s", "min"),
        latencia_max_s=("latencia_s", "max"),
    )
)

faixas_plausiveis["amplitude_aprovacao"] = (
    faixas_plausiveis["aprovacao_max"] - faixas_plausiveis["aprovacao_min"]
)
faixas_plausiveis["amplitude_cobertura"] = (
    faixas_plausiveis["cobertura_max"] - faixas_plausiveis["cobertura_min"]
)
faixas_plausiveis["amplitude_fidelidade"] = (
    faixas_plausiveis["fidelidade_max"] - faixas_plausiveis["fidelidade_min"]
)
faixas_plausiveis["amplitude_latencia_s"] = (
    faixas_plausiveis["latencia_max_s"] - faixas_plausiveis["latencia_min_s"]
)

display(faixas_plausiveis.round(4))


,id,aprovacao_min,aprovacao_max,cobertura_min,cobertura_max,fidelidade_min,fidelidade_max,latencia_min_s,latencia_max_s,amplitude_aprovacao,amplitude_cobertura,amplitude_fidelidade,amplitude_latencia_s
0,T01,1.0,1.0,1.0000,1.0000,1.0,1.0,3.491,5.044,0.0,0.0,0.0,1.553
1,T02,0.0,0.0,0.5000,0.5000,1.0,1.0,3.695,4.176,0.0,0.0,0.0,0.481
2,T03,1.0,1.0,1.0000,1.0000,0.0,1.0,4.010,4.601,0.0,0.0,1.0,0.591
3,T04,1.0,1.0,1.0000,1.0000,0.0,0.0,4.067,5.038,0.0,0.0,0.0,0.971
4,T05,1.0,1.0,1.0000,1.0000,1.0,1.0,3.096,4.136,0.0,0.0,0.0,1.040
5,T06,1.0,1.0,1.0000,1.0000,1.0,1.0,2.928,3.536,0.0,0.0,0.0,0.608
6,T07,1.0,1.0,1.0000,1.0000,1.0,1.0,3.680,5.506,0.0,0.0,0.0,1.826
7,T08,0.0,0.0,0.6667,0.6667,0.0,0.0,5.872,7.154,0.0,0.0,0.0,1.282
8,T10,1.0,1.0,NaN,NaN,NaN,NaN,1.902,2.569,0.0,NaN,NaN,0.667
9,T11,1.0,1.0,NaN,NaN,NaN,NaN,4.237,5.218,0.0,NaN,NaN,0.981


### F.2 Sobreposição entre faixas

Para evitar uma comparação arbitrária entre métricas diferentes, a sobreposição é verificada **entre casos, para a mesma métrica**. Dois casos apresentam faixas sobrepostas quando os intervalos `[mínimo, máximo]` possuem interseção.

Como aprovação é binária e cobertura/fidelidade podem assumir poucos valores neste benchmark, a sobreposição é usada como um diagnóstico: quando dois casos têm intervalos sobrepostos, diferenças pontuais entre eles podem ser menores que a variabilidade observada e não devem sustentar, sozinhas, uma afirmação de superioridade.


In [35]:
# ============================================================
# F.2 PARES DE CASOS COM FAIXAS SOBREPOSTAS
# ============================================================

from itertools import combinations


metricas_faixa = {
    "aprovacao": ("aprovacao_min", "aprovacao_max"),
    "cobertura": ("cobertura_min", "cobertura_max"),
    "fidelidade": ("fidelidade_min", "fidelidade_max"),
}

pares_sobrepostos = []

for (i, a), (j, b) in combinations(faixas_plausiveis.iterrows(), 2):
    for metrica, (col_min, col_max) in metricas_faixa.items():
        if intervalos_sobrepostos(
            a[col_min], a[col_max], b[col_min], b[col_max]
        ):
            pares_sobrepostos.append({
                "caso_a": a["id"],
                "caso_b": b["id"],
                "metrica": metrica,
                "faixa_a": f"[{a[col_min]:.2f}, {a[col_max]:.2f}]",
                "faixa_b": f"[{b[col_min]:.2f}, {b[col_max]:.2f}]",
            })

df_pares_sobrepostos = pd.DataFrame(pares_sobrepostos)

print("Pares de casos com pelo menos uma faixa sobreposta:",
      df_pares_sobrepostos[["caso_a", "caso_b"]].drop_duplicates().shape[0])

display(df_pares_sobrepostos.head(30))


Pares de casos com pelo menos uma faixa sobreposta: 45


,caso_a,caso_b,metrica,faixa_a,faixa_b
0,T01,T02,fidelidade,"[1.00, 1.00]","[1.00, 1.00]"
1,T01,T03,aprovacao,"[1.00, 1.00]","[1.00, 1.00]"
2,T01,T03,cobertura,"[1.00, 1.00]","[1.00, 1.00]"
3,T01,T03,fidelidade,"[1.00, 1.00]","[0.00, 1.00]"
4,T01,T04,aprovacao,"[1.00, 1.00]","[1.00, 1.00]"
5,T01,T04,cobertura,"[1.00, 1.00]","[1.00, 1.00]"
6,T01,T05,aprovacao,"[1.00, 1.00]","[1.00, 1.00]"
7,T01,T05,cobertura,"[1.00, 1.00]","[1.00, 1.00]"
8,T01,T05,fidelidade,"[1.00, 1.00]","[1.00, 1.00]"
9,T01,T06,aprovacao,"[1.00, 1.00]","[1.00, 1.00]"


In [36]:
# ============================================================
# F.3 SINAL, RUÍDO E INSTABILIDADE
# ============================================================

resumo_faixas = faixas_plausiveis.copy()

resumo_faixas["instabilidade_resultado"] = (
    (resumo_faixas["amplitude_aprovacao"] > 0)
    | (resumo_faixas["amplitude_cobertura"] > 0)
    | (resumo_faixas["amplitude_fidelidade"] > 0)
)

resumo_faixas["apenas_ruido_latencia"] = (
    (~resumo_faixas["instabilidade_resultado"])
    & (resumo_faixas["amplitude_latencia_s"] > 0)
)

display(
    resumo_faixas[
        [
            "id",
            "amplitude_aprovacao",
            "amplitude_cobertura",
            "amplitude_fidelidade",
            "amplitude_latencia_s",
            "instabilidade_resultado",
            "apenas_ruido_latencia",
        ]
    ].round(4)
)


,id,amplitude_aprovacao,amplitude_cobertura,amplitude_fidelidade,amplitude_latencia_s,instabilidade_resultado,apenas_ruido_latencia
0,T01,0.0,0.0,0.0,1.553,False,True
1,T02,0.0,0.0,0.0,0.481,False,True
2,T03,0.0,0.0,1.0,0.591,True,False
3,T04,0.0,0.0,0.0,0.971,False,True
4,T05,0.0,0.0,0.0,1.040,False,True
5,T06,0.0,0.0,0.0,0.608,False,True
6,T07,0.0,0.0,0.0,1.826,False,True
7,T08,0.0,0.0,0.0,1.282,False,True
8,T10,0.0,NaN,NaN,0.667,False,True
9,T11,0.0,NaN,NaN,0.981,False,True


### F.4 Interpretação

A análise acima impede que uma diferença pontual seja automaticamente tratada como melhoria real. Quando a faixa de um caso varia entre rodadas, essa amplitude representa a variabilidade observada no próprio experimento. Quando duas faixas se sobrepõem, a comparação entre esses casos deve ser interpretada com cautela.

A latência é tratada separadamente: variação apenas de tempo, sem alteração de aprovação, cobertura ou fidelidade, é classificada como **ruído operacional de latência**, e não como mudança de capacidade do sistema. Já alterações em aprovação, cobertura ou fidelidade são sinais de instabilidade relevante para a confiabilidade.

Essa análise complementa D: D demonstra repetibilidade por rodadas; F transforma essa repetição em limites empíricos explícitos e evita conclusões de superioridade baseadas apenas em um valor isolado.


In [37]:
# ============================================================
# F.4 RESUMO CONSOLIDADO POR CASO
# Compatibilidade robusta com G, H e J
# ============================================================

# IMPORTANTE:
# df_confiabilidade é a fonte efetivamente produzida em D.
# Esta célula não presume colunas que não estejam nesse DataFrame.


# Agregações obrigatórias: somente colunas existentes em D.
agregacoes_f = {
    "tipo": ("tipo", lambda s: _primeiro_valido(s, "")),
    "verificacao": ("verificacao", lambda s: _primeiro_valido(s, "")),
    "aprovacao_media": (
        "aprovado",
        lambda s: s.dropna().astype(float).mean()
        if s.notna().any() else np.nan
    ),
    "cobertura_media": ("cobertura", "mean"),
    "fidelidade_media": ("fidelidade_evidencias", "mean"),
    "abstencao_media": (
        "informacao_insuficiente",
        lambda s: s.dropna().astype(float).mean()
        if s.notna().any() else np.nan
    ),
    "latencia_media_s": ("latencia_s", "mean"),
    "respostas_distintas": ("resposta", lambda s: s.fillna("").nunique()),
    "erros_operacionais": ("erro", lambda s: s.notna().sum()),
}

# Só acrescenta dimensões se a coluna realmente existir.
if "dimensoes" in df_confiabilidade.columns:
    agregacoes_f["classificacoes_distintas"] = (
        "dimensoes",
        lambda s: s.astype(str).nunique()
    )

resumo_f = (
    df_confiabilidade
    .groupby("id", as_index=False)
    .agg(**agregacoes_f)
)

# Se D não materializou dimensões, a métrica fica explicitamente indisponível,
# em vez de provocar erro ou inventar um valor.
if "classificacoes_distintas" not in resumo_f.columns:
    resumo_f["classificacoes_distintas"] = np.nan

# Recupera amplitudes calculadas em F.1.
resumo_f = resumo_f.merge(
    faixas_plausiveis[
        [
            "id",
            "amplitude_aprovacao",
            "amplitude_cobertura",
            "amplitude_fidelidade",
            "amplitude_latencia_s",
        ]
    ],
    on="id",
    how="left",
)


resumo_f["classificacao_sinal"] = resumo_f.apply(
    lambda row: classificar_sinal(
        row, CRITERIOS_SUCESSO["fidelidade_evidencias"]
    ),
    axis=1
)

# Variação de redação é observável diretamente.
resumo_f["variacao_redacao"] = resumo_f["respostas_distintas"] > 1

# Variação de classificação só é afirmada se D tiver registrado dimensões.
resumo_f["variacao_classificacao"] = (
    resumo_f["classificacoes_distintas"].fillna(1) > 1
)

resumo_f["variacao_aprovacao"] = (
    resumo_f["amplitude_aprovacao"].fillna(0) > 0
)
resumo_f["erro_operacional"] = resumo_f["erros_operacionais"] > 0

resumo_f["variacao_relevante"] = (
    resumo_f["variacao_aprovacao"]
    | resumo_f["variacao_classificacao"]
    | resumo_f["erro_operacional"]
)

resumo_f["ruido_operacional_latencia"] = (
    (~resumo_f["variacao_relevante"])
    & (resumo_f["amplitude_latencia_s"].fillna(0) > 0)
)

print("Colunas disponíveis em df_confiabilidade:")
print(df_confiabilidade.columns.tolist())

print("\nResumo consolidado usado por G:")
display(
    resumo_f[
        [
            "id",
            "tipo",
            "verificacao",
            "aprovacao_media",
            "cobertura_media",
            "fidelidade_media",
            "abstencao_media",
            "classificacao_sinal",
            "variacao_relevante",
            "ruido_operacional_latencia",
        ]
    ].round(4)
)


Colunas disponíveis em df_confiabilidade:
['id', 'grupo', 'dimensao', 'tipo', 'verificacao', 'pergunta', 'esperado', 'resposta', 'evidencias_declaradas', 'documento_declarado', 'confianca', 'informacao_insuficiente', 'avaliacao', 'cobertura', 'aprovado', 'abstencao_correta', 'possui_evidencias', 'total_evidencias_declaradas', 'total_evidencias_validadas', 'fidelidade_evidencias', 'todas_evidencias_validadas', 'detalhes_validacao_evidencias', 'dimensoes_detectadas', 'documentos_consultados', 'secoes_recuperadas', 'evidencias_recuperadas', 'contexto_recuperado', 'evidencia_suficiente', 'justificativa_suficiencia', 'latencia_s', 'chamadas_llm', 'chamadas_ferramentas', 'passos', 'erro', 'rodada']

Resumo consolidado usado por G:


,id,tipo,verificacao,aprovacao_media,cobertura_media,fidelidade_media,abstencao_media,classificacao_sinal,variacao_relevante,ruido_operacional_latencia
0,T01,normal,auto,1.0,1.0000,1.0000,0.0,Sem sinal crítico,False,True
1,T02,normal,auto,0.0,0.5000,1.0000,1.0,Limitação sistemática,False,True
2,T03,normal,auto,1.0,1.0000,0.5556,0.0,Correção sem fidelidade suficiente,False,True
3,T04,normal,auto,1.0,1.0000,0.0000,0.0,Correção sem fidelidade suficiente,False,True
4,T05,normal,auto,1.0,1.0000,1.0000,0.0,Sem sinal crítico,False,True
5,T06,normal,auto,1.0,1.0000,1.0000,0.0,Sem sinal crítico,False,True
6,T07,normal,auto,1.0,1.0000,1.0000,0.0,Sem sinal crítico,False,True
7,T08,composto,auto,0.0,0.6667,0.0000,0.0,Limitação sistemática,False,True
8,T09,interpretativo,manual,NaN,NaN,1.0000,0.0,Sem sinal crítico,False,False
9,T10,informacao_ausente,auto,1.0,NaN,NaN,1.0,Contenção positiva,False,True


# G. Avaliação Ética: Assimetria entre erros

Atribua uma gravidade a cada tipo de erro do seu domínio, justificando pelo dano concreto. Não
existe valor correto; existe valor declarado e defendido.


## G.1 Princípio da assimetria e escala de gravidade

Em um sistema de consulta sobre dados de vigilância epidemiológica, erros diferentes não têm o mesmo impacto. Uma resposta apresentada como fundamentada sem suporte verificável pode induzir interpretação epidemiológica inadequada, enquanto uma abstenção explícita diante de evidência insuficiente preserva a incerteza.

A gravidade é atribuída ao **comportamento do sistema**, não ao tema clínico da pergunta. Para tornar essa política auditável, adota-se uma escala ordinal de **1 a 5**:

| Comportamento observado | Gravidade | Dano concreto que justifica o peso |
|---|---:|---|
| Resposta automática incorreta ou incompleta | **5** | Pode produzir conclusão inadequada apresentada como resposta válida. |
| Resposta aprovada, mas com fidelidade textual abaixo do limiar | **5** | Pode aparentar fundamentação que não foi validada no contexto recuperado. |
| Desempenho instável entre rodadas | **4** | A mesma condição experimental pode produzir decisões diferentes, reduzindo previsibilidade. |
| Caso manual com fidelidade abaixo do limiar | **4** | A fundamentação é insuficiente para aceitação automática e exige inspeção humana. |
| Avaliação manual sem decisão automática | **2** | Há incerteza residual, mas o sistema não deve convertê-la automaticamente em conclusão válida. |
| Abstenção quando a evidência é insuficiente | **1** | É comportamento protetivo: preserva a incerteza e evita extrapolação. |
| Resposta aprovada com fidelidade adequada | **1** | Não foi identificado sinal crítico pelos critérios adotados. |

Os pesos são deliberadamente assimétricos: **afirmar algo incorreto ou insuficientemente fundamentado recebe peso maior do que reconhecer explicitamente a insuficiência de informação**. A escala é uma política de risco deste experimento, e não uma medida universal de dano clínico.


In [38]:
# ============================================================
# G.2 APLICAÇÃO DA GRAVIDADE AOS CASOS T01–T14
# ============================================================

import pandas as pd
import numpy as np

df_etica_g = resumo_f.copy()


avaliacao_g = df_etica_g.apply(
    lambda row: avaliar_assimetria(
        row, CRITERIOS_SUCESSO["fidelidade_evidencias"]
    ),
    axis=1,
)
df_etica_g = pd.concat([df_etica_g, avaliacao_g], axis=1)

display(
    df_etica_g[
        [
            "id",
            "tipo",
            "verificacao",
            "aprovacao_media",
            "cobertura_media",
            "fidelidade_media",
            "abstencao_media",
            "categoria_etica",
            "severidade",
            "gravidade",
            "acao_recomendada",
        ]
    ]
)


,id,tipo,verificacao,aprovacao_media,cobertura_media,fidelidade_media,abstencao_media,categoria_etica,severidade,gravidade,acao_recomendada
0,T01,normal,auto,1.0,1.000000,1.000000,0.0,Sem sinal crítico automático,Baixa,1,Manter monitoramento e rastreabilidade.
1,T02,normal,auto,0.0,0.500000,1.000000,1.0,Resposta incorreta ou incompleta,Alta,5,Não usar a resposta sem revisão; investigar recuperação/geração.
2,T03,normal,auto,1.0,1.000000,0.555556,0.0,Correção sem fidelidade suficiente,Alta,5,Exigir evidência verificável antes de aceitar a resposta.
3,T04,normal,auto,1.0,1.000000,0.000000,0.0,Correção sem fidelidade suficiente,Alta,5,Exigir evidência verificável antes de aceitar a resposta.
4,T05,normal,auto,1.0,1.000000,1.000000,0.0,Sem sinal crítico automático,Baixa,1,Manter monitoramento e rastreabilidade.
5,T06,normal,auto,1.0,1.000000,1.000000,0.0,Sem sinal crítico automático,Baixa,1,Manter monitoramento e rastreabilidade.
6,T07,normal,auto,1.0,1.000000,1.000000,0.0,Sem sinal crítico automático,Baixa,1,Manter monitoramento e rastreabilidade.
7,T08,composto,auto,0.0,0.666667,0.000000,0.0,Resposta incorreta ou incompleta,Alta,5,Não usar a resposta sem revisão; investigar recuperação/geração.
8,T09,interpretativo,manual,NaN,NaN,1.000000,0.0,Avaliação manual necessária,Indeterminada,2,Não inferir aprovação ou reprovação automaticamente.
9,T10,informacao_ausente,auto,1.0,NaN,NaN,1.0,Abstenção protetiva,Baixa,1,Manter a abstenção e explicitar a insuficiência de evidência.


## G.3 Ordenação simples e ordenação ponderada

A **ordenação simples** considera apenas quantos casos de cada categoria ética foram observados. A **ordenação ponderada** considera também a gravidade atribuída, usando:

\[
	ext{pontuação ponderada} =
\sum_{i=1}^{n} 	ext{gravidade}_i
\]

Assim, uma categoria pouco frequente, mas de gravidade alta, pode receber prioridade maior do que uma categoria mais frequente de baixo impacto. A comparação não estima risco populacional; ela prioriza os comportamentos observados neste conjunto experimental.


In [39]:
# ============================================================
# G.3 COMPARAÇÃO: ORDENAÇÃO SIMPLES × PONDERADA
# ============================================================

ordenacao_g = (
    df_etica_g
    .groupby("categoria_etica", as_index=False)
    .agg(
        frequencia=("id", "count"),
        gravidade_media=("gravidade", "mean"),
        pontuacao_ponderada=("gravidade", "sum"),
    )
)

ordenacao_simples_g = (
    ordenacao_g
    .sort_values(
        ["frequencia", "gravidade_media"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)
ordenacao_simples_g["posicao_simples"] = (
    ordenacao_simples_g.index + 1
)

ordenacao_ponderada_g = (
    ordenacao_g
    .sort_values(
        ["pontuacao_ponderada", "frequencia"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)
ordenacao_ponderada_g["posicao_ponderada"] = (
    ordenacao_ponderada_g.index + 1
)

comparacao_ordenacoes_g = (
    ordenacao_simples_g[
        [
            "categoria_etica",
            "frequencia",
            "gravidade_media",
            "posicao_simples",
        ]
    ]
    .merge(
        ordenacao_ponderada_g[
            [
                "categoria_etica",
                "pontuacao_ponderada",
                "posicao_ponderada",
            ]
        ],
        on="categoria_etica",
        how="outer",
    )
)

comparacao_ordenacoes_g["mudanca_posicao"] = (
    comparacao_ordenacoes_g["posicao_simples"]
    - comparacao_ordenacoes_g["posicao_ponderada"]
)

print("ORDENAÇÃO SIMPLES — frequência observada")
display(
    ordenacao_simples_g[
        [
            "posicao_simples",
            "categoria_etica",
            "frequencia",
            "gravidade_media",
        ]
    ]
)

print("\nORDENAÇÃO PONDERADA — frequência × gravidade")
display(
    ordenacao_ponderada_g[
        [
            "posicao_ponderada",
            "categoria_etica",
            "frequencia",
            "pontuacao_ponderada",
        ]
    ]
)

print("\nCOMPARAÇÃO DAS POSIÇÕES")
display(
    comparacao_ordenacoes_g.sort_values("posicao_ponderada")
)


ORDENAÇÃO SIMPLES — frequência observada


,posicao_simples,categoria_etica,frequencia,gravidade_media
0,1,Sem sinal crítico automático,4,1.0
1,2,Correção sem fidelidade suficiente,3,5.0
2,3,Resposta incorreta ou incompleta,2,5.0
3,4,Avaliação manual necessária,2,2.0
4,5,Abstenção protetiva,2,1.0
5,6,Fundamentação requer inspeção humana,1,4.0



ORDENAÇÃO PONDERADA — frequência × gravidade


,posicao_ponderada,categoria_etica,frequencia,pontuacao_ponderada
0,1,Correção sem fidelidade suficiente,3,15
1,2,Resposta incorreta ou incompleta,2,10
2,3,Sem sinal crítico automático,4,4
3,4,Avaliação manual necessária,2,4
4,5,Fundamentação requer inspeção humana,1,4
5,6,Abstenção protetiva,2,2



COMPARAÇÃO DAS POSIÇÕES


,categoria_etica,frequencia,gravidade_media,posicao_simples,pontuacao_ponderada,posicao_ponderada,mudanca_posicao
2,Correção sem fidelidade suficiente,3,5.0,2,15,1,1
4,Resposta incorreta ou incompleta,2,5.0,3,10,2,1
5,Sem sinal crítico automático,4,1.0,1,4,3,-2
1,Avaliação manual necessária,2,2.0,4,4,4,0
3,Fundamentação requer inspeção humana,1,4.0,6,4,5,1
0,Abstenção protetiva,2,1.0,5,2,6,-1


## G.4 Interpretação ética

A ordenação simples responde à pergunta **“qual comportamento apareceu mais vezes?”**. A ordenação ponderada responde a uma pergunta diferente: **“qual comportamento concentra maior carga de gravidade segundo a política declarada?”**.

Essa distinção é importante porque frequência e consequência não são equivalentes. Um comportamento protetivo ou de baixa gravidade pode ocorrer várias vezes sem constituir a principal prioridade de correção. Por outro lado, respostas incorretas ou aparentemente corretas sem fidelidade suficiente recebem peso 5 porque podem produzir uma conclusão apresentada como fundamentada sem que essa fundamentação tenha sido demonstrada pelo experimento.

A interpretação das posições deve ser feita somente após a execução da célula anterior. Mudanças entre as duas ordenações mostram exatamente o efeito da assimetria de dano introduzida pela ponderação.


## G.5 Síntese

A avaliação ética passa a combinar três elementos: **categoria do comportamento**, **gravidade explícita de 1 a 5** e **frequência observada**. Isso evita tratar todos os erros como equivalentes e torna transparente por que determinados comportamentos recebem prioridade arquitetural.

A ponderação não substitui as métricas de correção, cobertura, fidelidade ou estabilidade. Ela funciona como uma camada adicional de decisão: os resultados técnicos indicam o que ocorreu; a política de gravidade explicita por que algumas ocorrências merecem maior atenção.


# H. Avaliação Ética: Distribuição dos erros

Acrescente ao conjunto pelo menos um caso representando o perfil ou a situação menos usual do seu domínio, aquele com menor chance de estar bem representado nos dados de treinamento do modelo. Descreva por que escolheu esse perfil.


## H.1 Distribuição dos erros

Enquanto G avalia a assimetria de gravidade, esta seção investiga **onde os sinais de erro e risco se concentram**. A unidade principal é o caso T01–T14, complementada pelas 42 execuções de D.

A distribuição não representa uma taxa de erro em produção: o conjunto foi construído deliberadamente com casos normais, ausentes, compostos, interpretativos e adversariais.


In [40]:
# ============================================================
# H.1 DISTRIBUIÇÃO DOS ERROS E SINAIS DE RISCO
# ============================================================

df_distribuicao_h = df_etica_g[
    [
        "id", "tipo", "verificacao", "categoria_etica",
        "severidade", "gravidade", "aprovacao_media",
        "cobertura_media", "fidelidade_media", "abstencao_media",
    ]
].copy()

df_erros_por_tipo = (
    df_distribuicao_h
    .groupby(["tipo", "severidade"], dropna=False, as_index=False)
    .agg(casos=("id", "nunique"))
    .sort_values(["tipo", "casos"], ascending=[True, False])
)

display(df_erros_por_tipo)

categorias_atencao = {
    "Resposta incorreta ou incompleta",
    "Correção sem fidelidade suficiente",
    "Desempenho instável",
    "Fundamentação requer inspeção humana",
}

df_sinais_atencao_h = df_distribuicao_h[
    df_distribuicao_h["categoria_etica"].isin(categorias_atencao)
].copy()

print("Casos com sinal de atenção:")
display(
    df_sinais_atencao_h.sort_values(
        ["gravidade", "tipo", "id"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
)

total_casos = df_distribuicao_h["id"].nunique()
total_atencao = df_sinais_atencao_h["id"].nunique()
total_protetivos = (
    df_distribuicao_h["categoria_etica"].eq("Abstenção protetiva").sum()
)

print("\nTotal de casos:", total_casos)
print("Casos com sinal de atenção:", total_atencao)
print("Casos com abstenção protetiva:", int(total_protetivos))
print("Proporção com sinal de atenção:", f"{total_atencao / total_casos:.2%}")


,tipo,severidade,casos
0,calculo_comparacao,Alta,1
1,composto,Alta,1
2,consulta_multidimensional,Média–Alta,1
3,informacao_ausente,Baixa,2
4,interpretativo,Indeterminada,1
6,normal,Baixa,4
5,normal,Alta,3
7,pressuposto_falso,Indeterminada,1


Casos com sinal de atenção:


,id,tipo,verificacao,categoria_etica,severidade,gravidade,aprovacao_media,cobertura_media,fidelidade_media,abstencao_media
0,T12,calculo_comparacao,auto,Correção sem fidelidade suficiente,Alta,5,1.0,1.000000,0.000000,0.0
1,T08,composto,auto,Resposta incorreta ou incompleta,Alta,5,0.0,0.666667,0.000000,0.0
2,T02,normal,auto,Resposta incorreta ou incompleta,Alta,5,0.0,0.500000,1.000000,1.0
3,T03,normal,auto,Correção sem fidelidade suficiente,Alta,5,1.0,1.000000,0.555556,0.0
4,T04,normal,auto,Correção sem fidelidade suficiente,Alta,5,1.0,1.000000,0.000000,0.0
5,T14,consulta_multidimensional,manual,Fundamentação requer inspeção humana,Média–Alta,4,NaN,NaN,0.000000,0.0



Total de casos: 14
Casos com sinal de atenção: 6
Casos com abstenção protetiva: 2
Proporção com sinal de atenção: 42.86%


## H.2 Caso de perfil/situação menos usual

Foi acrescentado um caso específico sobre **Turvânia (GO)**. A escolha não pressupõe que seus habitantes constituam um “perfil raro”; o aspecto menos usual é a **granularidade geográfica**: trata-se de um município pequeno e específico, muito menos provável de estar representado no conhecimento paramétrico geral de um LLM do que informações nacionais ou de grandes estados.

Isso torna o caso útil para verificar se a resposta depende da evidência recuperada da coleção semântica, em vez de conhecimento geral do modelo. O caso é executado **uma vez como diagnóstico adicional de H** e não é incorporado retrospectivamente às 42 execuções congeladas de D.


In [41]:
# ============================================================
# H.2 CASO ADICIONAL — SITUAÇÃO GEOGRÁFICA MENOS USUAL
# ============================================================

CASO_H_PERFIL_MENOS_USUAL = {
    "id": "H15",
    "grupo": "diagnostico_h",
    "dimensao": "geografica",
    "tipo": "perfil_menos_usual",
    "verificacao": "manual",
    "pergunta": (
        "O que as evidências disponíveis informam especificamente "
        "sobre a incidência de dengue em Turvânia, Goiás, em 2026?"
    ),
    # Avaliação manual: não se inventa um valor esperado novo para o benchmark.
    "esperado": [],
    "cobertura_minima": 1.0,
}

resultado_h15 = executar_caso(CASO_H_PERFIL_MENOS_USUAL)

campos_h15 = [
    "id", "pergunta", "resposta", "confianca",
    "fidelidade_evidencias", "informacao_insuficiente",
    "evidencias_declaradas", "documentos_consultados",
    "erro",
]

print("CASO H15 — PERFIL/SITUAÇÃO MENOS USUAL")
for campo in campos_h15:
    if campo in resultado_h15:
        print(f"\n{campo}:")
        print(resultado_h15[campo])

print("\nInterpretação:")
if resultado_h15.get("erro"):
    print("- O caso apresentou erro operacional e deve ser investigado.")
elif resultado_h15.get("informacao_insuficiente"):
    print(
        "- O sistema declarou insuficiência de informação. "
        "Para este caso diagnóstico, isso é preferível a completar "
        "o dado a partir de conhecimento paramétrico não rastreado."
    )
else:
    print(
        "- O sistema produziu resposta para um município específico. "
        "A aceitação substantiva permanece manual e deve considerar "
        "as evidências declaradas e sua fidelidade."
    )


CASO H15 — PERFIL/SITUAÇÃO MENOS USUAL

id:
H15

pergunta:
O que as evidências disponíveis informam especificamente sobre a incidência de dengue em Turvânia, Goiás, em 2026?

resposta:
Em 2026, Turvânia, Goiás, apresentou um total de 407 notificações de dengue. A população do município em 2022 era de 4.480 habitantes, resultando em uma incidência de 9.084,82 casos por 100 mil habitantes. O grau de criticidade da situação foi classificado como alta.

confianca:
alta

fidelidade_evidencias:
1.0

informacao_insuficiente:
False

evidencias_declaradas:
['Municipio: Turvânia | Uf: GO | Notificações: 407 | Populacao 2022: 4.480 | Incidencia 100mil: 9.084,82 | Grau criticidade: Alta']

documentos_consultados:
['sinan_dengue_2026_temporal_brasil.md', 'sinan_dengue_2026_geografico_brasil.md']

erro:
None

Interpretação:
- O sistema produziu resposta para um município específico. A aceitação substantiva permanece manual e deve considerar as evidências declaradas e sua fidelidade.


## H.3 Par mínimo executado: T01 × T11

O par mínimo selecionado reutiliza duas perguntas do conjunto congelado que **já foram executadas três vezes**:

- **T01:** “Quantas notificações de dengue foram consideradas no Brasil em 2026?”
- **T11:** “Quantas notificações de dengue ocorreram entre pessoas do sexo feminino no Brasil em 2026?”

A mudança mínima é a inclusão do recorte **“pessoas do sexo feminino”**. T01 é documentalmente respondível; T11 permanece no mesmo domínio, mas solicita uma estratificação que não está materializada na coleção semântica avaliada. Assim, o comportamento esperado muda de resposta factual para abstenção.

A comparação abaixo usa as execuções reais de D e considera **acerto, confiança, evidência e ressalvas**.


In [42]:
# ============================================================
# H.3 PAR MÍNIMO EXECUTADO — T01 × T11
# ============================================================



df_par_exec = df_confiabilidade[
    df_confiabilidade["id"].isin(["T01", "T11"])
].copy()

if "evidencias_declaradas" in df_par_exec.columns:
    df_par_exec["n_evidencias"] = (
        df_par_exec["evidencias_declaradas"].apply(contar_evidencias)
    )
else:
    df_par_exec["n_evidencias"] = np.nan

# A ressalva é operacionalizada como reconhecimento explícito de insuficiência.
if "informacao_insuficiente" in df_par_exec.columns:
    df_par_exec["ressalva_insuficiencia"] = (
        df_par_exec["informacao_insuficiente"].fillna(False).astype(bool)
    )
else:
    df_par_exec["ressalva_insuficiencia"] = (
        df_par_exec["abstencao_correta"].fillna(False).astype(bool)
    )

resumo_par_minimo = (
    df_par_exec
    .groupby("id", as_index=False)
    .agg(
        execucoes=("rodada", "count"),
        acerto_medio=("aprovado", lambda s: s.dropna().astype(float).mean()),
        confianca_modal=("confianca", modo_textual),
        fidelidade_media=("fidelidade_evidencias", "mean"),
        evidencias_min=("n_evidencias", "min"),
        evidencias_max=("n_evidencias", "max"),
        ressalva_insuficiencia=(
            "ressalva_insuficiencia",
            lambda s: bool(s.astype(bool).any())
        ),
        abstencao_correta=(
            "abstencao_correta",
            lambda s: (
                bool(s.dropna().astype(bool).all())
                if s.notna().any() else None
            )
        ),
    )
)

display(resumo_par_minimo)

print("\nLeitura do par:")
for _, row in resumo_par_minimo.iterrows():
    print(
        f"- {row['id']}: acerto={row['acerto_medio']:.2f}; "
        f"confiança modal={row['confianca_modal']}; "
        f"evidências={row['evidencias_min']}–{row['evidencias_max']}; "
        f"ressalva de insuficiência={row['ressalva_insuficiencia']}."
    )


,id,execucoes,acerto_medio,confianca_modal,fidelidade_media,evidencias_min,evidencias_max,ressalva_insuficiencia,abstencao_correta
0,T01,3,1.0,alta,1.0,2,2,False,None
1,T11,3,1.0,baixa,NaN,0,0,True,True



Leitura do par:
- T01: acerto=1.00; confiança modal=alta; evidências=2–2; ressalva de insuficiência=False.
- T11: acerto=1.00; confiança modal=baixa; evidências=0–0; ressalva de insuficiência=True.


## H.4 Diferença do par mínimo versus variação entre execuções

Para que a diferença entre T01 e T11 seja interpretada como **sinal**, e não apenas ruído entre rodadas, ela é comparada com a amplitude intracasos observada nas três execuções.

O teste não exige que a redação seja idêntica. Ele verifica se as propriedades relevantes do par — aprovação, confiança, quantidade de evidências e reconhecimento de insuficiência — mudam de maneira consistente entre as duas perguntas e se essa mudança é maior do que a variação observada dentro de cada caso.


In [43]:
# ============================================================
# H.4 PAR MÍNIMO × VARIAÇÃO ENTRE RODADAS
# ============================================================

ordem_confianca = {"baixa": 1, "media": 2, "média": 2, "alta": 3}

df_par_var = df_par_exec.copy()
df_par_var["confianca_num"] = (
    df_par_var["confianca"]
    .astype(str)
    .str.lower()
    .map(ordem_confianca)
)

variacao_intracaso = (
    df_par_var
    .groupby("id", as_index=False)
    .agg(
        aprovacao_min=("aprovado", lambda s: s.dropna().astype(float).min()),
        aprovacao_max=("aprovado", lambda s: s.dropna().astype(float).max()),
        confianca_min=("confianca_num", "min"),
        confianca_max=("confianca_num", "max"),
        evidencias_min=("n_evidencias", "min"),
        evidencias_max=("n_evidencias", "max"),
        ressalva_min=("ressalva_insuficiencia", "min"),
        ressalva_max=("ressalva_insuficiencia", "max"),
    )
)

display(variacao_intracaso)

t01 = resumo_par_minimo.set_index("id").loc["T01"]
t11 = resumo_par_minimo.set_index("id").loc["T11"]

conf_t01 = ordem_confianca.get(str(t01["confianca_modal"]).lower(), np.nan)
conf_t11 = ordem_confianca.get(str(t11["confianca_modal"]).lower(), np.nan)

comparacao_par_variacao = pd.DataFrame([
    {
        "propriedade": "acerto",
        "diferenca_entre_T01_T11": abs(
            float(t01["acerto_medio"]) - float(t11["acerto_medio"])
        ),
        "maior_amplitude_intracaso": max(
            (variacao_intracaso["aprovacao_max"]
             - variacao_intracaso["aprovacao_min"]).fillna(0)
        ),
    },
    {
        "propriedade": "confiança",
        "diferenca_entre_T01_T11": abs(conf_t01 - conf_t11),
        "maior_amplitude_intracaso": max(
            (variacao_intracaso["confianca_max"]
             - variacao_intracaso["confianca_min"]).fillna(0)
        ),
    },
    {
        "propriedade": "número de evidências",
        "diferenca_entre_T01_T11": abs(
            float(t01["evidencias_min"]) - float(t11["evidencias_min"])
        ),
        "maior_amplitude_intracaso": max(
            (variacao_intracaso["evidencias_max"]
             - variacao_intracaso["evidencias_min"]).fillna(0)
        ),
    },
    {
        "propriedade": "ressalva de insuficiência",
        "diferenca_entre_T01_T11": int(
            bool(t01["ressalva_insuficiencia"])
            != bool(t11["ressalva_insuficiencia"])
        ),
        "maior_amplitude_intracaso": max(
            (
                variacao_intracaso["ressalva_max"].astype(int)
                - variacao_intracaso["ressalva_min"].astype(int)
            )
        ),
    },
])

comparacao_par_variacao["diferenca_supera_variacao"] = (
    comparacao_par_variacao["diferenca_entre_T01_T11"]
    > comparacao_par_variacao["maior_amplitude_intracaso"]
)

display(comparacao_par_variacao)

print(
    "\nConclusão diagnóstica: propriedades com diferença entre o par "
    "maior que a variação intracasos =",
    int(comparacao_par_variacao["diferenca_supera_variacao"].sum()),
    "de",
    len(comparacao_par_variacao),
)


,id,aprovacao_min,aprovacao_max,confianca_min,confianca_max,evidencias_min,evidencias_max,ressalva_min,ressalva_max
0,T01,1.0,1.0,3,3,2,2,False,False
1,T11,1.0,1.0,1,1,0,0,True,True


,propriedade,diferenca_entre_T01_T11,maior_amplitude_intracaso,diferenca_supera_variacao
0,acerto,0.0,0.0,False
1,confiança,2.0,0.0,True
2,número de evidências,2.0,0.0,True
3,ressalva de insuficiência,1.0,0.0,True



Conclusão diagnóstica: propriedades com diferença entre o par maior que a variação intracasos = 3 de 4


## H.5 Síntese

H acrescenta duas verificações complementares. O caso H15 testa uma consulta geograficamente específica, escolhida por ser menos provável de ser respondida de forma confiável apenas pelo conhecimento paramétrico do modelo. O resultado é mantido como diagnóstico manual e não altera o benchmark congelado.

O par T01×T11, por sua vez, é quantitativo porque ambas as perguntas foram executadas nas três rodadas. A comparação permite verificar se uma pequena mudança semântica — acrescentar uma estratificação não materializada — produz mudança consistente no comportamento do sistema. A tabela de H.4 explicita se essa diferença excede a própria variação entre execuções, evitando interpretar simples flutuação como sensibilidade semântica.


# I. Rastreabilidade

**[PREENCHER EM MARKDOWN]**

Marque o que o seu registro permite hoje, e o que faltaria:

- [ ] o registro sobrevive à execução, em arquivo e não só em memória;
- [ ] é possível identificar qual componente produziu cada afirmação;
- [ ] as decisões de roteamento estão registradas com justificativa;
- [ ] as chamadas a ferramentas estão registradas com argumentos;
- [ ] é possível mostrar ao usuário o trecho da fonte que sustenta a resposta;
- [ ] é possível reconstruir uma execução específica dias depois.

## I.1 Definição e objetivo

Nesta entrega, **rastreabilidade** é a capacidade de reconstruir o caminho entre a pergunta e a avaliação final da resposta. A cadeia considerada é:

**pergunta → dimensões detectadas → documentos/seções recuperados → contexto fornecido ao LLM → evidências declaradas → validação textual das evidências → resposta → avaliação.**

A análise reutiliza os traces registrados em `df_confiabilidade`. Portanto, esta seção não executa novamente o agente e não altera os resultados experimentais das seções anteriores.


In [44]:
# ============================================================
# I.2 INVENTÁRIO DE RASTREABILIDADE
# ============================================================

import pandas as pd

campos_trace = [
    "pergunta",
    "dimensoes_detectadas",
    "documentos_consultados",
    "secoes_recuperadas",
    "contexto_recuperado",
    "evidencias_declaradas",
    "detalhes_validacao_evidencias",
    "resposta",
    "aprovado",
    "fidelidade_evidencias",
    "erro",
]

presenca_campos = pd.DataFrame({
    "campo": campos_trace,
    "presente_no_dataframe": [
        campo in df_confiabilidade.columns
        for campo in campos_trace
    ],
})

display(presenca_campos)

campos_ausentes = presenca_campos.loc[
    ~presenca_campos["presente_no_dataframe"],
    "campo"
].tolist()

if campos_ausentes:
    print("⚠️ Campos de rastreabilidade ausentes:", campos_ausentes)
else:
    print("✓ Todos os campos previstos para a cadeia de rastreabilidade estão disponíveis.")


,campo,presente_no_dataframe
0,pergunta,True
1,dimensoes_detectadas,True
2,documentos_consultados,True
3,secoes_recuperadas,True
4,contexto_recuperado,True
5,evidencias_declaradas,True
6,detalhes_validacao_evidencias,True
7,resposta,True
8,aprovado,True
9,fidelidade_evidencias,True


✓ Todos os campos previstos para a cadeia de rastreabilidade estão disponíveis.


In [45]:
# ============================================================
# I.3 CADEIA DE RASTREABILIDADE POR EXECUÇÃO
# ============================================================


df_trace_i = df_confiabilidade.copy()

df_trace_i["qtd_documentos_consultados"] = (
    df_trace_i["documentos_consultados"].apply(tamanho_seguro)
)
df_trace_i["qtd_evidencias_declaradas"] = (
    df_trace_i["evidencias_declaradas"].apply(tamanho_seguro)
)
df_trace_i["possui_contexto_recuperado"] = (
    df_trace_i["contexto_recuperado"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)
df_trace_i["possui_dimensao_detectada"] = (
    df_trace_i["dimensoes_detectadas"].apply(tamanho_seguro) > 0
)
df_trace_i["possui_resposta"] = (
    df_trace_i["resposta"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

# A cadeia é considerada estruturalmente completa quando há os
# elementos mínimos para reconstruir o caminho. Em abstenções,
# evidências declaradas podem legitimamente ser vazias.
df_trace_i["trace_estrutural_completo"] = (
    df_trace_i["pergunta"].fillna("").astype(str).str.strip().ne("")
    & df_trace_i["possui_dimensao_detectada"]
    & df_trace_i["possui_contexto_recuperado"]
    & df_trace_i["possui_resposta"]
    & df_trace_i["erro"].isna()
)

display(
    df_trace_i[
        [
            "rodada",
            "id",
            "dimensoes_detectadas",
            "qtd_documentos_consultados",
            "qtd_evidencias_declaradas",
            "fidelidade_evidencias",
            "informacao_insuficiente",
            "aprovado",
            "trace_estrutural_completo",
        ]
    ]
)

print("\nRESUMO")
print("Execuções totais:", len(df_trace_i))
print(
    "Traces estruturalmente completos:",
    int(df_trace_i["trace_estrutural_completo"].sum()),
)
print(
    "Proporção de traces completos:",
    f"{df_trace_i['trace_estrutural_completo'].mean():.2%}",
)


,rodada,id,dimensoes_detectadas,qtd_documentos_consultados,qtd_evidencias_declaradas,fidelidade_evidencias,informacao_insuficiente,aprovado,trace_estrutural_completo
0,1,T01,[geral],1,2,1.000000,False,True,True
1,1,T02,[temporal],1,1,1.000000,True,False,True
2,1,T03,"[geografica, geral]",2,1,1.000000,False,True,True
3,1,T04,[geografica],1,1,0.000000,False,True,True
4,1,T05,[virologica],1,1,1.000000,False,True,True
5,1,T06,[clinica],1,2,1.000000,False,True,True
6,1,T07,[desfechos],2,4,1.000000,False,True,True
7,1,T08,"[temporal, geografica, clinica]",3,7,0.000000,False,False,True
8,1,T09,[virologica],1,2,1.000000,False,None,True
9,1,T10,[clinica],1,0,NaN,True,True,True



RESUMO
Execuções totais: 42
Traces estruturalmente completos: 42
Proporção de traces completos: 100.00%


## I.4 Inspeção de uma cadeia completa

A célula seguinte seleciona automaticamente uma execução com resposta não abstida e exibe os principais elementos da cadeia. O objetivo não é avaliar novamente o conteúdo, mas demonstrar que é possível reconstruir **como a resposta foi produzida e posteriormente auditada**.


In [46]:
# ============================================================
# I.4 EXEMPLO AUDITÁVEL DE TRACE
# ============================================================

candidatos_trace = df_trace_i[
    df_trace_i["trace_estrutural_completo"]
    & (~df_trace_i["informacao_insuficiente"].fillna(False))
].copy()

if candidatos_trace.empty:
    print("Nenhuma execução não abstida com trace completo foi encontrada.")
else:
    exemplo_trace = candidatos_trace.iloc[0]

    print("=" * 70)
    print(
        f"TRACE AUDITÁVEL — {exemplo_trace['id']} "
        f"(rodada {exemplo_trace['rodada']})"
    )
    print("=" * 70)

    print("\n1. PERGUNTA")
    print(exemplo_trace["pergunta"])

    print("\n2. DIMENSÕES DETECTADAS")
    print(exemplo_trace["dimensoes_detectadas"])

    print("\n3. DOCUMENTOS CONSULTADOS")
    print(exemplo_trace["documentos_consultados"])

    print("\n4. SEÇÕES RECUPERADAS")
    print(exemplo_trace["secoes_recuperadas"])

    print("\n5. CONTEXTO RECUPERADO — trecho inicial")
    contexto = str(exemplo_trace["contexto_recuperado"])
    print(contexto[:1500] + ("..." if len(contexto) > 1500 else ""))

    print("\n6. EVIDÊNCIAS DECLARADAS")
    print(exemplo_trace["evidencias_declaradas"])

    print("\n7. VALIDAÇÃO DAS EVIDÊNCIAS")
    print(exemplo_trace["detalhes_validacao_evidencias"])

    print("\n8. RESPOSTA")
    print(exemplo_trace["resposta"])

    print("\n9. AVALIAÇÃO")
    print({
        "aprovado": exemplo_trace["aprovado"],
        "cobertura": exemplo_trace["cobertura"],
        "fidelidade_evidencias": exemplo_trace["fidelidade_evidencias"],
        "confianca": exemplo_trace["confianca"],
    })


TRACE AUDITÁVEL — T01 (rodada 1)

1. PERGUNTA
Quantas notificações de dengue foram consideradas no Brasil em 2026?

2. DIMENSÕES DETECTADAS
['geral']

3. DOCUMENTOS CONSULTADOS
['sinan_dengue_2026_panorama_geral.md']

4. SEÇÕES RECUPERADAS
{'sinan_dengue_2026_panorama_geral.md': ['Situação geral', 'Interpretação dos resultados']}

5. CONTEXTO RECUPERADO — trecho inicial
DOCUMENTO: sinan_dengue_2026_panorama_geral
DIMENSÃO: geral

## Situação geral

- **Total de notificações:** 444.266
- **Período:** semanas epidemiológicas 1 a 34

## Interpretação dos resultados

No período entre as semanas epidemiológicas 1 e 34 de 2026, foram consideradas 444.266 notificações de dengue no SINAN. Foram registradas, em média, 13.066,65 notificações de dengue por semana. O maior volume ocorreu na semana epidemiológica 15, com 21.470 notificações, correspondendo a 4,83% do total do período. Na distribuição geográfica, GO apresentou o maior número de notificações segundo residência, com 108.912 notificaçõ

In [47]:
# ============================================================
# I.5 LACUNAS DE RASTREABILIDADE
# ============================================================

# Casos em que a resposta foi aprovada automaticamente, mas a
# fidelidade ficou abaixo do limiar, merecem atenção especial:
# o resultado final pode estar correto, porém a trilha de
# fundamentação não é suficientemente verificável.

df_lacunas_i = df_trace_i[
    (df_trace_i["aprovado"] == True)
    & df_trace_i["fidelidade_evidencias"].notna()
    & (
        df_trace_i["fidelidade_evidencias"]
        < CRITERIOS_SUCESSO["fidelidade_evidencias"]
    )
][
    [
        "rodada",
        "id",
        "aprovado",
        "cobertura",
        "fidelidade_evidencias",
        "documentos_consultados",
        "evidencias_declaradas",
    ]
].copy()

print("EXECUÇÕES APROVADAS COM FIDELIDADE ABAIXO DO LIMIAR")
display(df_lacunas_i.reset_index(drop=True))

print(
    "\nTotal:",
    len(df_lacunas_i),
)


EXECUÇÕES APROVADAS COM FIDELIDADE ABAIXO DO LIMIAR


,rodada,id,aprovado,cobertura,fidelidade_evidencias,documentos_consultados,evidencias_declaradas
0,1,T04,True,1.0,0.000000,[sinan_dengue_2026_geografico_brasil.md],"[**Codigo municipio:** 5221502 | **Municipio:** Turvânia | **Uf:** GO | **Incidencia 100mil:** 9.084,82 | **Grau criticidade:** Alta]"
1,1,T12,True,1.0,0.000000,[sinan_dengue_2026_clinico_brasil.md],"[Percentual sinal clinico maior frequencia: Febre - 88,21%, Percentual sim para Mialgia - 79,12%]"
2,2,T03,True,1.0,0.000000,"[sinan_dengue_2026_geografico_brasil.md, sinan_dengue_2026_panorama_geral.md]","[No período entre as semanas epidemiológicas 1 e 34 de 2026, foram consideradas 444.266 notificações de dengue no SINAN. Na distribuição geográfica, GO apresentou o maior número de notificações segundo residência, com 108.912 notificações.]"
3,2,T04,True,1.0,0.000000,[sinan_dengue_2026_geografico_brasil.md],"[**Codigo municipio:** 5221502 | **Municipio:** Turvânia | **Uf:** GO | **Incidencia 100mil:** 9.084,82 | **Grau criticidade:** Alta]"
4,2,T12,True,1.0,0.000000,[sinan_dengue_2026_clinico_brasil.md],"[Percentual sinal clinico maior frequencia: Febre - 88,21%, Percentual sim para Mialgia - 79,12%]"
5,3,T03,True,1.0,0.666667,"[sinan_dengue_2026_geografico_brasil.md, sinan_dengue_2026_panorama_geral.md]","[No período entre as semanas epidemiológicas 1 e 34 de 2026, foram consideradas 444.266 notificações de dengue no SINAN. Na distribuição geográfica, GO apresentou o maior número de notificações segundo residência, com 108.912 notificações., - **UF com maior número de notificações por residência:** GO, - **Total de notificações nessa UF:** 108.912]"
6,3,T04,True,1.0,0.000000,[sinan_dengue_2026_geografico_brasil.md],"[**Codigo municipio:** 5221502 | **Municipio:** Turvânia | **Uf:** GO | **Incidencia 100mil:** 9.084,82 | **Grau criticidade:** Alta]"
7,3,T12,True,1.0,0.000000,[sinan_dengue_2026_clinico_brasil.md],"[Percentual sinal clinico maior frequencia: Febre - 88,21%, Percentual sim Mialgia: 79,12%]"



Total: 8


## I.6 Síntese

A arquitetura registra elementos suficientes para reconstruir a maior parte da cadeia entre pergunta, recuperação, geração e avaliação. Essa rastreabilidade é importante porque permite distinguir **acerto do resultado** de **qualidade da fundamentação**.

A existência de um trace estrutural não garante, por si só, que a resposta esteja corretamente fundamentada. Por isso, a trilha é combinada com a validação textual das evidências. Execuções aprovadas com fidelidade abaixo do limiar constituem uma lacuna de fundamentação: o sistema chegou ao conteúdo esperado pelo benchmark, mas a evidência declarada não pôde ser confirmada adequadamente no contexto recuperado.

Assim, a rastreabilidade é tratada como uma propriedade necessária para auditoria, mas não suficiente para demonstrar correção ou confiabilidade. Ela fornece a infraestrutura para localizar onde uma falha ocorreu — classificação, recuperação, contexto, evidência ou geração — e sustenta a análise de riscos arquiteturais da seção J.


# J. Matriz de consequências e riscos da arquitetura

Uma linha por modo de falha observado. A coluna final força a coerência com os pesos da seção G.


## J.1 Objetivo e coerência com G

A matriz consolida os riscos identificados ao longo de B–I e utiliza **a mesma escala de gravidade 1–5 definida em G**. Isso evita manter uma escala qualitativa independente para a priorização.

O peso representa a consequência potencial do modo de falha caso a contenção não funcione. A eficácia observada de uma mitigação é registrada separadamente em `risco_residual`; por exemplo, informação ausente recebe gravidade alta porque uma resposta inventada teria consequência importante, embora T10 e T11 tenham demonstrado contenção adequada por abstenção.


In [48]:
# ============================================================
# J.2 MATRIZ DE CONSEQUÊNCIAS COERENTE COM OS PESOS DE G
# ============================================================

import pandas as pd

riscos_j = [
    {
        "risco": "Recuperação ou composição insuficiente",
        "consequencia": "Resposta incorreta ou incompleta apresentada como válida.",
        "evidencia_experimental": "T02 e comportamento composto observado em T08",
        "gravidade": 5,
        "justificativa_peso": (
            "Equivale ao comportamento de resposta automática incorreta "
            "ou incompleta, classificado com peso 5 em G."
        ),
        "mitigacao_atual": (
            "Verificação de suficiência, recuperação complementar "
            "e abstenção quando necessário."
        ),
        "risco_residual": (
            "A verificação de suficiência pode não detectar toda "
            "incompletude semântica."
        ),
    },
    {
        "risco": "Evidência declarada não validada textualmente",
        "consequencia": (
            "Resposta pode parecer fundamentada sem suporte textual "
            "validado no contexto recuperado."
        ),
        "evidencia_experimental": "T03, T04, T12 e lacunas de fidelidade em I.5",
        "gravidade": 5,
        "justificativa_peso": (
            "Corresponde à categoria 'Correção sem fidelidade suficiente', "
            "que recebe peso 5 em G."
        ),
        "mitigacao_atual": "Validação textual e limiar explícito de fidelidade.",
        "risco_residual": (
            "A métrica textual pode rejeitar paráfrases semanticamente corretas."
        ),
    },
    {
        "risco": "Instabilidade entre execuções",
        "consequencia": (
            "A mesma condição experimental pode produzir decisões diferentes."
        ),
        "evidencia_experimental": "Variações identificadas em D e F",
        "gravidade": 4,
        "justificativa_peso": (
            "Corresponde à categoria 'Desempenho instável', peso 4 em G."
        ),
        "mitigacao_atual": (
            "Execuções repetidas, temperatura zero, trace e análise de faixas."
        ),
        "risco_residual": (
            "Configuração determinística não garante identidade absoluta."
        ),
    },
    {
        "risco": "Generalização indevida de evidência específica",
        "consequencia": (
            "Achado de um subconjunto pode ser apresentado como válido "
            "para toda a população."
        ),
        "evidencia_experimental": "T13 e análise de pressuposto falso",
        "gravidade": 4,
        "justificativa_peso": (
            "Exige inspeção interpretativa humana; alinhado aos casos "
            "manuais de maior consequência, peso 4 em G."
        ),
        "mitigacao_atual": (
            "Suficiência, uso das evidências e inspeção de pressupostos."
        ),
        "risco_residual": (
            "A detecção depende da interpretação semântica da pergunta."
        ),
    },
    {
        "risco": "Cálculo derivado sem fundamentação adequada",
        "consequencia": (
            "Resultado numérico pode ser apresentado sem trilha suficiente "
            "para sustentar a derivação."
        ),
        "evidencia_experimental": "T12",
        "gravidade": 5,
        "justificativa_peso": (
            "Pode produzir resposta aparentemente correta sem fundamentação "
            "validada; alinhado ao peso 5 de G."
        ),
        "mitigacao_atual": (
            "Regra de não realizar cálculo derivado sem suporte "
            "e auditoria de fidelidade."
        ),
        "risco_residual": (
            "O LLM ainda pode executar implicitamente uma operação derivada."
        ),
    },
    {
        "risco": "Informação ausente na coleção semântica",
        "consequencia": (
            "Se a insuficiência não for detectada, o sistema pode produzir "
            "resposta não fundamentada."
        ),
        "evidencia_experimental": "T10, T11 e par mínimo T01×T11",
        "gravidade": 5,
        "justificativa_peso": (
            "O dano potencial é resposta não fundamentada, peso 5; "
            "a abstenção observada é registrada como mitigação, não como "
            "redução do peso inerente."
        ),
        "mitigacao_atual": (
            "Abstenção estruturada, confiança baixa, evidência vazia "
            "e comunicação explícita de insuficiência."
        ),
        "risco_residual": (
            "T10/T11 mostraram contenção adequada; permanece risco se "
            "a detecção de insuficiência falhar."
        ),
    },
    {
        "risco": "Trace completo interpretado como prova de correção",
        "consequencia": (
            "Auditabilidade pode ser confundida com correção ou fidelidade."
        ),
        "evidencia_experimental": (
            "I.3: traces completos; I.5: execuções aprovadas abaixo "
            "do limiar de fidelidade."
        ),
        "gravidade": 2,
        "justificativa_peso": (
            "É um risco de interpretação que exige revisão, mas não equivale "
            "por si só a uma resposta automática incorreta; alinhado ao peso 2."
        ),
        "mitigacao_atual": (
            "Separação entre rastreabilidade, correção, cobertura e fidelidade."
        ),
        "risco_residual": (
            "Usuários ainda podem superinterpretar logs e evidências declaradas."
        ),
    },
]

df_riscos_j = pd.DataFrame(riscos_j)

rotulo_gravidade = {
    5: "Alta",
    4: "Média–Alta",
    3: "Média",
    2: "Baixa–Média",
    1: "Baixa",
}
df_riscos_j["severidade"] = df_riscos_j["gravidade"].map(rotulo_gravidade)

display(
    df_riscos_j[
        [
            "risco", "consequencia", "evidencia_experimental",
            "gravidade", "severidade", "justificativa_peso",
            "mitigacao_atual", "risco_residual",
        ]
    ]
)

print("\nDISTRIBUIÇÃO DOS RISCOS POR PESO")
display(
    df_riscos_j["gravidade"]
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("gravidade")
    .reset_index(name="riscos")
)


,risco,consequencia,evidencia_experimental,gravidade,severidade,justificativa_peso,mitigacao_atual,risco_residual
0,Recuperação ou composição insuficiente,Resposta incorreta ou incompleta apresentada como válida.,T02 e comportamento composto observado em T08,5,Alta,"Equivale ao comportamento de resposta automática incorreta ou incompleta, classificado com peso 5 em G.","Verificação de suficiência, recuperação complementar e abstenção quando necessário.",A verificação de suficiência pode não detectar toda incompletude semântica.
1,Evidência declarada não validada textualmente,Resposta pode parecer fundamentada sem suporte textual validado no contexto recuperado.,"T03, T04, T12 e lacunas de fidelidade em I.5",5,Alta,"Corresponde à categoria 'Correção sem fidelidade suficiente', que recebe peso 5 em G.",Validação textual e limiar explícito de fidelidade.,A métrica textual pode rejeitar paráfrases semanticamente corretas.
2,Instabilidade entre execuções,A mesma condição experimental pode produzir decisões diferentes.,Variações identificadas em D e F,4,Média–Alta,"Corresponde à categoria 'Desempenho instável', peso 4 em G.","Execuções repetidas, temperatura zero, trace e análise de faixas.",Configuração determinística não garante identidade absoluta.
3,Generalização indevida de evidência específica,Achado de um subconjunto pode ser apresentado como válido para toda a população.,T13 e análise de pressuposto falso,4,Média–Alta,"Exige inspeção interpretativa humana; alinhado aos casos manuais de maior consequência, peso 4 em G.","Suficiência, uso das evidências e inspeção de pressupostos.",A detecção depende da interpretação semântica da pergunta.
4,Cálculo derivado sem fundamentação adequada,Resultado numérico pode ser apresentado sem trilha suficiente para sustentar a derivação.,T12,5,Alta,Pode produzir resposta aparentemente correta sem fundamentação validada; alinhado ao peso 5 de G.,Regra de não realizar cálculo derivado sem suporte e auditoria de fidelidade.,O LLM ainda pode executar implicitamente uma operação derivada.
5,Informação ausente na coleção semântica,"Se a insuficiência não for detectada, o sistema pode produzir resposta não fundamentada.","T10, T11 e par mínimo T01×T11",5,Alta,"O dano potencial é resposta não fundamentada, peso 5; a abstenção observada é registrada como mitigação, não como redução do peso inerente.","Abstenção estruturada, confiança baixa, evidência vazia e comunicação explícita de insuficiência.",T10/T11 mostraram contenção adequada; permanece risco se a detecção de insuficiência falhar.
6,Trace completo interpretado como prova de correção,Auditabilidade pode ser confundida com correção ou fidelidade.,I.3: traces completos; I.5: execuções aprovadas abaixo do limiar de fidelidade.,2,Baixa–Média,"É um risco de interpretação que exige revisão, mas não equivale por si só a uma resposta automática incorreta; alinhado ao peso 2.","Separação entre rastreabilidade, correção, cobertura e fidelidade.",Usuários ainda podem superinterpretar logs e evidências declaradas.



DISTRIBUIÇÃO DOS RISCOS POR PESO


,gravidade,riscos
0,5,4
1,4,2
2,2,1


In [49]:
# ============================================================
# J.3 PRIORIZAÇÃO DOS RISCOS PELO MESMO PESO 1–5 DE G
# ============================================================

df_priorizacao_j = (
    df_riscos_j
    .sort_values(
        ["gravidade", "risco"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

df_priorizacao_j["prioridade"] = df_priorizacao_j.index + 1

display(
    df_priorizacao_j[
        [
            "prioridade",
            "risco",
            "gravidade",
            "severidade",
            "evidencia_experimental",
            "mitigacao_atual",
            "risco_residual",
        ]
    ]
)

assert df_priorizacao_j["gravidade"].between(1, 5).all()
assert df_priorizacao_j["gravidade"].is_monotonic_decreasing

print(
    "\nCoerência verificada: J utiliza diretamente a escala 1–5 "
    "definida e justificada em G."
)


,prioridade,risco,gravidade,severidade,evidencia_experimental,mitigacao_atual,risco_residual
0,1,Cálculo derivado sem fundamentação adequada,5,Alta,T12,Regra de não realizar cálculo derivado sem suporte e auditoria de fidelidade.,O LLM ainda pode executar implicitamente uma operação derivada.
1,2,Evidência declarada não validada textualmente,5,Alta,"T03, T04, T12 e lacunas de fidelidade em I.5",Validação textual e limiar explícito de fidelidade.,A métrica textual pode rejeitar paráfrases semanticamente corretas.
2,3,Informação ausente na coleção semântica,5,Alta,"T10, T11 e par mínimo T01×T11","Abstenção estruturada, confiança baixa, evidência vazia e comunicação explícita de insuficiência.",T10/T11 mostraram contenção adequada; permanece risco se a detecção de insuficiência falhar.
3,4,Recuperação ou composição insuficiente,5,Alta,T02 e comportamento composto observado em T08,"Verificação de suficiência, recuperação complementar e abstenção quando necessário.",A verificação de suficiência pode não detectar toda incompletude semântica.
4,5,Generalização indevida de evidência específica,4,Média–Alta,T13 e análise de pressuposto falso,"Suficiência, uso das evidências e inspeção de pressupostos.",A detecção depende da interpretação semântica da pergunta.
5,6,Instabilidade entre execuções,4,Média–Alta,Variações identificadas em D e F,"Execuções repetidas, temperatura zero, trace e análise de faixas.",Configuração determinística não garante identidade absoluta.
6,7,Trace completo interpretado como prova de correção,2,Baixa–Média,I.3: traces completos; I.5: execuções aprovadas abaixo do limiar de fidelidade.,"Separação entre rastreabilidade, correção, cobertura e fidelidade.",Usuários ainda podem superinterpretar logs e evidências declaradas.



Coerência verificada: J utiliza diretamente a escala 1–5 definida e justificada em G.


## J.4 Interpretação arquitetural

A priorização de J passa a usar diretamente os pesos de G. Os riscos de peso **5** são aqueles capazes de produzir resposta incorreta, incompleta ou aparentemente fundamentada sem suporte verificável. Peso **4** identifica comportamentos que exigem atenção elevada, como instabilidade e interpretação semântica sensível. O peso **2** é reservado aqui ao risco de superinterpretar a rastreabilidade como prova de correção.

A matriz também separa **gravidade inerente** de **eficácia da mitigação**. Informação ausente, por exemplo, continua sendo risco potencial de peso 5, mas T10/T11 demonstram que a abstenção pode reduzir a consequência observada. Portanto, uma mitigação bem-sucedida não apaga a gravidade do modo de falha que ela procura conter.


## J.5 Síntese

Com a escala unificada, G e J deixam de utilizar critérios paralelos. G atribui os pesos aos comportamentos observados; J aplica a mesma lógica aos modos de falha arquiteturais e registra, separadamente, evidência experimental, mitigação e risco residual.

Isso permite que a recomendação final seja fundamentada não apenas em frequência ou correção, mas também na consequência potencial dos modos de falha e na capacidade demonstrada de contenção.


# K. Recomendação final

**[PREENCHER EM MARKDOWN]**

> **Qual versão o grupo levaria para uso real?**

Informe:

1. o critério otimizado: qualidade, custo, latência ou previsibilidade;
2. a evidência principal que sustenta a escolha;
3. as ressalvas, isto é, o que os dados não permitem afirmar;
4. que evidência ainda faltaria, e como vocês a obteriam.

A resposta pode ser uma versão intermediária, ou o próprio baseline. O que precisa estar sustentado
é o raciocínio.

## K.1 Recomendação

Com base nos resultados obtidos, a recomendação é **manter a arquitetura como protótipo experimental auditável, com uso assistido e supervisão humana**, e não como sistema autônomo para apoiar decisões epidemiológicas.

A arquitetura demonstrou capacidades relevantes: abstenção diante de informação ausente, registro de traces completos, recuperação orientada por dimensões e mecanismos explícitos para avaliar correção, cobertura e fidelidade textual. Entretanto, os experimentos também identificaram riscos residuais de alta prioridade relacionados a recuperação/composição insuficiente, fundamentação não validada textualmente, cálculo derivado e informação ausente caso a contenção falhe.


## K.2 Evidências que sustentam a recomendação

A recomendação decorre do conjunto de resultados das seções anteriores:

- **Contenção:** T10 e T11 apresentaram abstenção correta quando a evidência necessária não estava disponível.
- **Confiabilidade:** as execuções repetidas mostraram que temperatura zero não elimina completamente a variabilidade, com destaque para consultas compostas como T08.
- **Fundamentação:** T03, T04 e T12 mostraram que uma resposta pode satisfazer o benchmark e, ainda assim, não atingir o limiar de fidelidade textual.
- **Assimetria ética:** responder de forma incorreta ou com fundamentação não verificável foi considerado mais grave do que explicitar insuficiência de informação.
- **Rastreabilidade:** as 42 execuções apresentaram trace estrutural completo, permitindo reconstruir o caminho entre pergunta, recuperação, resposta e avaliação; contudo, trace completo não implica correção ou fidelidade.
- **Riscos arquiteturais:** a matriz de J mostra que os mecanismos existentes reduzem riscos, mas não os eliminam.


## K.3 Mecanismos que devem ser preservados

A evolução da arquitetura deve preservar cinco propriedades já demonstradas no protótipo: **abstenção explícita**, **verificação de suficiência**, **rastreabilidade da recuperação e geração**, **validação das evidências** e **avaliação por múltiplas dimensões**, evitando reduzir desempenho a uma única taxa de acerto.

Também deve ser mantida a separação entre ausência de evidência e erro. Uma abstenção correta não deve ser penalizada da mesma maneira que uma resposta não fundamentada.


## K.4 Melhorias prioritárias

Antes de considerar maior autonomia, as prioridades são:

1. fortalecer a recuperação e composição de evidências para consultas temporais, compostas e multidimensionais;
2. tornar a validação de evidências mais robusta, complementando a correspondência textual com mecanismos capazes de avaliar equivalência semântica;
3. tratar cálculos derivados por mecanismo determinístico ou ferramenta específica, registrando operandos e operação no trace;
4. fortalecer a detecção de pressupostos falsos e de generalizações entre subconjuntos e população total;
5. ampliar o conjunto de testes e repetir a avaliação em uma amostra mais representativa de perguntas reais, pois T01–T14 foi construído deliberadamente para incluir casos difíceis e não estima uma taxa de erro em produção.


## K.5 Conclusão final

O experimento mostra que a arquitetura é **viável para investigação e uso assistido**, principalmente porque combina geração em linguagem natural com mecanismos de contenção e auditoria. Os resultados, porém, não sustentam uso autônomo em um contexto de vigilância epidemiológica.

A principal conclusão do Entregável 4 é que **correção da resposta, isoladamente, não é suficiente para caracterizar confiabilidade**. Uma arquitetura desse tipo deve ser avaliada também pela capacidade de reconhecer ausência de informação, manter evidências verificáveis, limitar extrapolações, apresentar comportamento estável e permitir reconstrução do processo que levou à resposta.

Assim, a recomendação final é continuar o desenvolvimento da arquitetura como sistema experimental de apoio, mantendo supervisão humana e priorizando as melhorias de fundamentação, composição de evidências e tratamento de operações derivadas antes de qualquer aumento de autonomia.


---

# Checklist antes da entrega

- [x] Estrutura herdada reunida no início; origem de cada resultado declarada.
- [x] Falhas plausíveis listadas, com o mecanismo que trata cada uma.
- [x] Pelo menos três mecanismos de contenção implementados e demonstrados.
- [x] Resposta degradada se identifica como degradada.
- [x] Pelo menos duas verificações que convertem falha silenciosa em detectável.
- [x] Conjunto congelado executado pelo menos três vezes, com a faixa relatada.
- [x] Casos instáveis identificados e comentados.
- [x] Tabela consolidada das quatro versões, com qualidade e custo.
- [x] Faixas plausíveis calculadas; pares sobrepostos identificados.
- [x] Afirmações de superioridade ancoradas em casos específicos e no trace.
- [x] Pesos de gravidade atribuídos e justificados pelo dano concreto.
- [x] Ordenação simples e ponderada comparadas.
- [x] Caso do perfil menos usual acrescentado e o resultado comentado.
- [x] Pelo menos um par mínimo executado, comparando acerto, confiança, evidência e ressalvas.
- [x] Diferença do par mínimo comparada com a variação entre execuções.
- [x] Rastreabilidade avaliada.
- [x] Matriz de consequências coerente com os pesos.
- [x] Recomendação final com critério, evidência, ressalvas e lacunas.
- [x] Notebook executado do início ao fim e salvo com as saídas.
- [x] Nenhuma chave de API no notebook.
